In [ ]:
#installations
!pip install biopython
!pip install circlify
!pip install pyahocorasick

In [ ]:
# ============================================================
# MERGE TWO FASTA FILES
# ============================================================

# Input FASTA files
fasta1 = "uniprotkb_sprot.fasta"
fasta2 = "trembl_protein_struc.fasta"

# Output merged FASTA
output_file = "complete_uniprot_prot.fasta"

print("Merging FASTA files...")

with open(output_file, "w") as outfile:

    for fasta in [fasta1, fasta2]:

        with open(fasta, "r") as infile:

            for line in infile:
                outfile.write(line)

print(f"Done!\nMerged FASTA saved as: {output_file}")

In [ ]:
from Bio import SeqIO

input_fasta = "complete_uniprot_prot.fasta"
isoform_fasta = "isoforms.fasta"
non_isoform_fasta = "non_isoforms.fasta"

isoforms = []
non_isoforms = []

for record in SeqIO.parse(input_fasta, "fasta"):
    header = record.description.lower()

    if "isoform" in header:
        isoforms.append(record)
    else:
        non_isoforms.append(record)

SeqIO.write(isoforms, isoform_fasta, "fasta")
SeqIO.write(non_isoforms, non_isoform_fasta, "fasta")

print(f"Isoforms: {len(isoforms)}")
print(f"Non-isoforms: {len(non_isoforms)}")

In [ ]:
# removing redundancy and getting accession

import csv
from tqdm import tqdm

# ================= SETTINGS =================
INPUT_FASTA = "non_isoforms.fasta"

# ONLY OUTPUT
OUTPUT_CSV = "non_isoform.csv"
# ============================================

# Store unique sequences
seen_sequences = set()

# Store unique UniProt IDs
output_rows = []

total_entries = 0
unique_entries = 0
redundant_entries = 0

# ===== READ FASTA =====
with open(INPUT_FASTA, "r") as f:

    current_header = None
    seq_parts = []

    for line in tqdm(f, desc="Reading FASTA"):

        line = line.strip()

        if line.startswith(">"):

            # Process previous entry
            if current_header is not None:

                total_entries += 1

                sequence = "".join(seq_parts)

                if sequence not in seen_sequences:

                    seen_sequences.add(sequence)
                    unique_entries += 1

                    # Extract UniProt ID
                    parts = current_header.split("|")

                    if len(parts) >= 2:
                        uniprot_id = parts[1]
                    else:
                        uniprot_id = current_header[1:].split()[0]

                    output_rows.append({
                        "accession_number": uniprot_id
                    })

                else:
                    redundant_entries += 1

            # Start next entry
            current_header = line
            seq_parts = []

        else:
            seq_parts.append(line)

    # ===== PROCESS LAST ENTRY =====
    if current_header is not None:

        total_entries += 1

        sequence = "".join(seq_parts)

        if sequence not in seen_sequences:

            seen_sequences.add(sequence)
            unique_entries += 1

            parts = current_header.split("|")

            if len(parts) >= 2:
                uniprot_id = parts[1]
            else:
                uniprot_id = current_header[1:].split()[0]

            output_rows.append({
                "accession_number": uniprot_id
            })

        else:
            redundant_entries += 1

# ===== SAVE CSV =====
with open(OUTPUT_CSV, "w", newline="") as csvfile:

    writer = csv.DictWriter(
        csvfile,
        fieldnames=["accession_number"]
    )

    writer.writeheader()
    writer.writerows(output_rows)

# ===== SUMMARY =====
print("\n===== SUMMARY =====")
print(f"Total FASTA entries:     {total_entries:,}")
print(f"Unique sequences kept:   {unique_entries:,}")
print(f"Redundant removed:       {redundant_entries:,}")

print(f"\nCSV saved → {OUTPUT_CSV}")

In [ ]:
# id mapping + pre-processing

import csv
import os
import sqlite3
import time
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import requests
from tqdm import tqdm

# ================= SETTINGS =================
INPUT_CSV         = "non_isoform.csv"

# FINAL OUTPUT (ONLY COMPLETE DATA)
OUTPUT_CSV        = "non_isoform_comp.csv"

PROGRESS_DB       = "progress.db"
COLUMN            = "accession_number"

TOTAL_CHUNKS      = 4

UNIPROT_BATCH     = 100
UNIPROT_WORKERS   = 5
UNIPROT_DELAY     = 0.2

NCBI_BATCH        = 200
NCBI_WORKERS      = 8
NCBI_DELAY        = 0.4

TIMEOUT           = 60
MAX_RETRIES       = 5

# Values treated as missing protein IDs
NO_PROTEIN = {"", "not_found", "error", "n/a", "none", "nan", "-"}
# ============================================

OUTPUT_COLUMNS = [
    "UniProt_ID",
    "GenBank_Nucleotide_ID",
    "GenBank_Protein_ID",
    "Molecule_Type",
    "CDS_Completeness",
    "UniProt_Protein_Length",
    "NCBI_Protein_Length",
]

thread_local = threading.local()
db_lock = threading.Lock()
csv_lock = threading.Lock()

# Global NCBI rate limiter
ncbi_rate_lock = threading.Lock()
ncbi_last_req = [0.0]
NCBI_MIN_GAP = 0.11


# ─────────────────────────── SESSION ───────────────────────────
def get_session():
    if not hasattr(thread_local, "session"):
        s = requests.Session()
        s.mount(
            "https://",
            requests.adapters.HTTPAdapter(
                pool_connections=20,
                pool_maxsize=20
            )
        )
        thread_local.session = s
    return thread_local.session


# ─────────────────────────── PROGRESS DB ───────────────────────
def init_db(path):
    con = sqlite3.connect(path, check_same_thread=False)

    con.execute("""
        CREATE TABLE IF NOT EXISTS done (
            uniprot_id TEXT PRIMARY KEY,
            nuc_id TEXT,
            prot_id TEXT,
            mol_type TEXT,
            completeness TEXT,
            uniprot_length INTEGER,
            ncbi_length INTEGER
        )
    """)

    con.commit()
    return con


def load_done(con):
    return {
        r[0]
        for r in con.execute(
            "SELECT uniprot_id FROM done"
        ).fetchall()
    }


def save_rows_db(con, rows):
    with db_lock:
        con.executemany(
            """
            INSERT OR REPLACE INTO done
            (
                uniprot_id,
                nuc_id,
                prot_id,
                mol_type,
                completeness,
                uniprot_length,
                ncbi_length
            )
            VALUES (?,?,?,?,?,?,?)
            """,
            [
                (
                    r["UniProt_ID"],
                    r["GenBank_Nucleotide_ID"],
                    r["GenBank_Protein_ID"],
                    r["Molecule_Type"],
                    r["CDS_Completeness"],
                    r["UniProt_Protein_Length"],
                    r["NCBI_Protein_Length"],
                )
                for r in rows
            ]
        )

        con.commit()


# ─────────────────────────── CSV ───────────────────────────────
def init_csv(path):
    if not os.path.exists(path):
        with open(path, "w", newline="") as f:
            csv.DictWriter(
                f,
                fieldnames=OUTPUT_COLUMNS
            ).writeheader()


def append_rows_csv(path, rows):
    with csv_lock:
        with open(path, "a", newline="") as f:
            writer = csv.DictWriter(
                f,
                fieldnames=OUTPUT_COLUMNS
            )

            for row in rows:
                writer.writerow(row)


# ─────────────────────────── HELPERS ───────────────────────────
def parse_completeness(raw):

    r = (raw or "").strip().lower()

    # =========================
    # MOLECULE TYPE
    # =========================
    if "mrna" in r:
        mol = "mRNA"

    elif "genomic" in r:
        mol = "Genomic_DNA"

    elif "wgs" in r:
        mol = "WGS"

    else:
        mol = "Unclassified"

    # =========================
    # COMPLETENESS CHECK
    # =========================
    incomplete_keywords = {
        "partial",
        "incomplete",
        "truncated",
        "fragment",
        "frameshift",
        "low quality",
        "low-quality"
    }

    complete = not any(
        keyword in r
        for keyword in incomplete_keywords
    )

    return mol, complete


def select_best(refs):
    if not refs:
        return None, None, "N/A", "N/A"

    priority = {
        "mRNA": 0,
        "Genomic_DNA": 1,
        "Other": 2
    }

    best = sorted(
        refs,
        key=lambda r: (
            0 if r["complete"] else 1,
            priority.get(r["mol"], 99)
        )
    )[0]

    return (
        best["nuc"],
        best["prot"],
        best["mol"],
        "Complete" if best["complete"] else "Partial"
    )


def parse_embl(entry):
    refs = []

    for db in entry.get("uniProtKBCrossReferences", []):

        if db.get("database") != "EMBL":
            continue

        nuc = db.get("id")
        prot = ""
        mol_raw = "Unknown"

        for p in db.get("properties", []):
            if p.get("key") == "ProteinId":
                prot = p.get("value", "")

            if p.get("key") == "MoleculeType":
                mol_raw = p.get("value", "Unknown")

        if not nuc or prot == "":
            continue

        mol, complete = parse_completeness(mol_raw)

        refs.append({
            "nuc": nuc,
            "prot": prot,
            "mol": mol,
            "complete": complete
        })

    return refs


def empty_rows(uids, status):
    return [
        {
            "UniProt_ID": uid,
            "GenBank_Nucleotide_ID": status,
            "GenBank_Protein_ID": status,
            "Molecule_Type": "N/A",
            "CDS_Completeness": "N/A",
            "UniProt_Protein_Length": None,
            "NCBI_Protein_Length": None,
        }
        for uid in uids
    ]


# ─────────────────────────── UNIPROT ───────────────────────────
def fetch_uniprot_batch(uid_batch):

    time.sleep(UNIPROT_DELAY)

    url = "https://rest.uniprot.org/uniprotkb/search"

    params = {
        "query": " OR ".join(
            f"accession:{u}" for u in uid_batch
        ),
        "format": "json",
        "size": len(uid_batch),
        "fields": "accession,sequence,xref_embl",
    }

    for attempt in range(MAX_RETRIES):

        try:
            r = get_session().get(
                url,
                params=params,
                timeout=TIMEOUT
            )

            if r.status_code == 429:
                wait = 2 ** (attempt + 2)
                time.sleep(wait)
                continue

            if r.status_code != 200:
                time.sleep(3)
                continue

            found = {}

            for entry in r.json().get("results", []):

                uid = entry.get("primaryAccession", "")
                length = entry.get(
                    "sequence",
                    {}
                ).get("length")

                refs = parse_embl(entry)

                nuc, prot, mol, comp = select_best(refs)

                found[uid] = {
                    "UniProt_ID": uid,
                    "GenBank_Nucleotide_ID": nuc,
                    "GenBank_Protein_ID": prot,
                    "Molecule_Type": mol,
                    "CDS_Completeness": comp,
                    "UniProt_Protein_Length": length,
                    "NCBI_Protein_Length": None,
                }

            return [
                found.get(
                    u,
                    {
                        "UniProt_ID": u,
                        "GenBank_Nucleotide_ID": "Not_Found",
                        "GenBank_Protein_ID": "Not_Found",
                        "Molecule_Type": "N/A",
                        "CDS_Completeness": "N/A",
                        "UniProt_Protein_Length": None,
                        "NCBI_Protein_Length": None,
                    }
                )
                for u in uid_batch
            ]

        except Exception:
            time.sleep(2 ** attempt)

    return empty_rows(uid_batch, "Error")


# ─────────────────────────── NCBI ──────────────────────────────
def fetch_ncbi_batch(prot_batch):

    with ncbi_rate_lock:

        now = time.monotonic()
        gap = NCBI_MIN_GAP - (now - ncbi_last_req[0])

        if gap > 0:
            time.sleep(gap)

        ncbi_last_req[0] = time.monotonic()

    bare_map = {}

    for uid, prot in prot_batch:
        bare_map[prot.split(".")[0]] = uid

    url = (
        "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
        f"?db=protein&id={','.join(bare_map.keys())}"
        f"&rettype=fasta&retmode=text"
        f"&api_key={NCBI_API_KEY}"
    )

    for attempt in range(MAX_RETRIES):

        try:
            r = get_session().get(
                url,
                timeout=TIMEOUT
            )

            if r.status_code == 429:
                wait = 2 ** (attempt + 2)
                time.sleep(wait)
                continue

            if r.status_code == 200:

                lengths = {}

                cur = None
                parts = []

                for line in r.text.splitlines():

                    if line.startswith(">"):

                        if cur:
                            lengths[cur] = len(
                                "".join(parts)
                            )

                        p = line[1:].split()

                        cur = (
                            p[0].split(".")[0]
                            if p else None
                        )

                        parts = []

                    else:
                        parts.append(line.strip())

                if cur:
                    lengths[cur] = len(
                        "".join(parts)
                    )

                return {
                    bare_map[bare]: lengths.get(bare)
                    for bare in bare_map
                }

            time.sleep(3)

        except Exception:
            time.sleep(2 ** attempt)

    return {
        bare_map[bare]: None
        for bare in bare_map
    }


# ─────────────────────────── PROCESS CHUNK ─────────────────────
def process_chunk(chunk_ids, chunk_num, con):

    skip = {
        "",
        "Not_Found",
        "Error",
        "N/A",
        "None",
        "nan",
        "none"
    }

    print(
        f"\n[Chunk {chunk_num}] "
        f"UniProt pass ({len(chunk_ids):,} IDs)"
    )

    batches = [
        chunk_ids[i:i + UNIPROT_BATCH]
        for i in range(
            0,
            len(chunk_ids),
            UNIPROT_BATCH
        )
    ]

    uni_rows = {}

    with ThreadPoolExecutor(
        max_workers=UNIPROT_WORKERS
    ) as ex:

        futures = {
            ex.submit(fetch_uniprot_batch, b): b
            for b in batches
        }

        for f in tqdm(
            as_completed(futures),
            total=len(futures),
            desc=f"Chunk {chunk_num} UniProt"
        ):

            for row in f.result():
                uni_rows[row["UniProt_ID"]] = row

    ncbi_pairs = [
        (uid, row["GenBank_Protein_ID"])
        for uid, row in uni_rows.items()
        if row.get("GenBank_Protein_ID")
        and str(
            row["GenBank_Protein_ID"]
        ).strip() not in skip
    ]

    print(
        f"[Chunk {chunk_num}] "
        f"NCBI pass ({len(ncbi_pairs):,} proteins)"
    )

    ncbi_lengths = {}

    if ncbi_pairs:

        batches = [
            ncbi_pairs[i:i + NCBI_BATCH]
            for i in range(
                0,
                len(ncbi_pairs),
                NCBI_BATCH
            )
        ]

        with ThreadPoolExecutor(
            max_workers=NCBI_WORKERS
        ) as ex:

            futures = {
                ex.submit(fetch_ncbi_batch, b): b
                for b in batches
            }

            for f in tqdm(
                as_completed(futures),
                total=len(futures),
                desc=f"Chunk {chunk_num} NCBI"
            ):
                ncbi_lengths.update(f.result())

    # ===== KEEP ONLY COMPLETE ROWS =====
    final_rows = []

    for uid in chunk_ids:

        row = uni_rows.get(
            uid,
            {
                "UniProt_ID": uid,
                "GenBank_Nucleotide_ID": "Error",
                "GenBank_Protein_ID": "Error",
                "Molecule_Type": "Error",
                "CDS_Completeness": "Error",
                "UniProt_Protein_Length": None,
                "NCBI_Protein_Length": None,
            }
        )

        row["NCBI_Protein_Length"] = ncbi_lengths.get(uid)

        prot_id = str(
            row["GenBank_Protein_ID"]
        ).strip().lower()

        # ONLY SAVE COMPLETE DATA
        if prot_id not in NO_PROTEIN:
            final_rows.append(row)

    if final_rows:
        save_rows_db(con, final_rows)
        append_rows_csv(OUTPUT_CSV, final_rows)

    print(
        f"[Chunk {chunk_num}] "
        f"Saved {len(final_rows):,} COMPLETE rows"
    )


# ─────────────────────────── MAIN ──────────────────────────────
def main():

    try:
        df = pd.read_csv(INPUT_CSV)

    except Exception as e:
        print(f"Error loading CSV: {e}")
        return

    all_ids = (
        df[COLUMN]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    total = len(all_ids)

    print(
        f"Total IDs: {total:,}"
    )

    con = init_db(PROGRESS_DB)

    done_set = load_done(con)

    init_csv(OUTPUT_CSV)

    print(
        f"Already done: {len(done_set):,}"
    )

    chunk_size = (
        total + TOTAL_CHUNKS - 1
    ) // TOTAL_CHUNKS

    chunks = [
        all_ids[i:i + chunk_size]
        for i in range(
            0,
            total,
            chunk_size
        )
    ]

    for idx, chunk in enumerate(chunks):

        todo = [
            uid
            for uid in chunk
            if uid not in done_set
        ]

        print(
            f"\n━━━ Chunk {idx+1}/{TOTAL_CHUNKS} "
            f"({len(todo):,} IDs) ━━━"
        )

        if not todo:
            print("Skipping")
            continue

        process_chunk(
            todo,
            idx + 1,
            con
        )

        done_set.update(todo)

        print(
            f"Progress: "
            f"{len(done_set):,}/{total:,}"
        )

    con.close()

    print(
        f"\n✓ COMPLETE DATA SAVED → "
        f"{OUTPUT_CSV}"
    )


if __name__ == "__main__":
    main()

In [ ]:
# cds extraction

import csv
import os
import sqlite3
import time
import threading
import random
from concurrent.futures import ThreadPoolExecutor, as_completed
from io import StringIO
import requests
from Bio import SeqIO
from tqdm import tqdm

# ================= SETTINGS =================
INPUT_CSV       = "non_isoform_comp.csv"
OUTPUT_FASTA    = "non_isoform_cds_output.fasta"
FAILED_CSV      = "non_isoform_cds_failed.csv"
PROGRESS_DB     = "cds_progress.db"

WORKERS         = 4              # ↓ reduced for stability
MIN_GAP         = 0.18           # ↓ safer global rate
TIMEOUT         = 90             # ↑ better for slow responses
MAX_RETRIES     = 5

NCBI_API_KEY    = "3ab472c6ad8f89a6b587fe4efc486db18309"
# ============================================

db_lock      = threading.Lock()
fasta_lock   = threading.Lock()
failed_lock  = threading.Lock()
rate_lock    = threading.Lock()
last_req     = [0.0]

FAILED_COLUMNS = ["UniProt_ID", "GenBank_Nucleotide_ID", "GenBank_Protein_ID", "Reason"]

thread_local = threading.local()

def get_session():
    if not hasattr(thread_local, "session"):
        s = requests.Session()
        s.mount("https://", requests.adapters.HTTPAdapter(
            pool_connections=20, pool_maxsize=20, max_retries=0))
        thread_local.session = s
    return thread_local.session


def throttle():
    with rate_lock:
        now = time.monotonic()
        gap = MIN_GAP - (now - last_req[0])
        if gap > 0:
            time.sleep(gap + random.uniform(0, 0.05))  # jitter added
        last_req[0] = time.monotonic()


# ─────────────────────────── DB ───────────────────────────
def init_db(path):
    con = sqlite3.connect(path, check_same_thread=False)
    con.execute("""
        CREATE TABLE IF NOT EXISTS done (
            uniprot_id TEXT PRIMARY KEY,
            status TEXT
        )
    """)
    con.commit()
    return con

def load_done(con):
    return {r[0] for r in con.execute("SELECT uniprot_id FROM done").fetchall()}

def mark_done(con, uid, status):
    with db_lock:
        con.execute("INSERT OR REPLACE INTO done VALUES (?,?)", (uid, status))
        con.commit()


# ─────────────────────────── OUTPUT ───────────────────────────
def init_fasta(path):
    if not os.path.exists(path):
        open(path, "w").close()

def write_fasta(path, row, seq):
    header = ">" + "|".join(str(v).split(".")[0] for v in row.values())
    with fasta_lock:
        with open(path, "a") as f:
            f.write(f"{header}\n{seq}\n")

def init_failed(path):
    if not os.path.exists(path):
        with open(path, "w", newline="") as f:
            csv.DictWriter(f, fieldnames=FAILED_COLUMNS).writeheader()

def write_failed(path, row):
    with failed_lock:
        with open(path, "a", newline="") as f:
            csv.DictWriter(f, fieldnames=FAILED_COLUMNS).writerow(row)


# ─────────────────────────── REQUEST ───────────────────────────
def ncbi_get(url):
    for attempt in range(MAX_RETRIES):
        try:
            throttle()
            r = get_session().get(url, timeout=TIMEOUT)

            if r.status_code == 429:
                wait = 2 ** (attempt + 2)
                tqdm.write(f"[429] sleeping {wait}s")
                time.sleep(wait)
                continue

            return r

        except Exception as e:
            if attempt < MAX_RETRIES - 1:
                tqdm.write(f"[conn err] retry {attempt+1}: {e}")
                time.sleep(2 ** attempt)
            else:
                raise

    raise Exception("Max retries exceeded")


# ─────────────────────────── FETCH ───────────────────────────
def fetch_via_protein(prot_id):
    bare = prot_id.split(".")[0]

    url = (
        "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
        f"?db=protein&id={bare}&rettype=fasta_cds_na&retmode=text"
        f"&api_key={NCBI_API_KEY}"
    )

    r = ncbi_get(url)

    if r.status_code != 200:
        return None, f"HTTP {r.status_code}"

    seq = "".join(
        line.strip() for line in r.text.splitlines()
        if line and not line.startswith(">")
    )

    if seq:
        return seq, None

    return None, "Empty sequence"


def fetch_via_nuccore(nuc_id, prot_id):
    bare = prot_id.split(".")[0]

    url = (
        "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
        f"?db=nuccore&id={nuc_id}&rettype=gb&retmode=text"
        f"&api_key={NCBI_API_KEY}"
    )

    r = ncbi_get(url)

    if r.status_code != 200:
        return None, f"HTTP {r.status_code}"

    try:
        record = SeqIO.read(StringIO(r.text), "genbank")
    except Exception as e:
        return None, f"Parse error: {e}"

    for f in record.features:
        if f.type == "CDS":
            for pid in f.qualifiers.get("protein_id", []):
                if pid.split(".")[0] == bare:
                    try:
                        return str(f.extract(record.seq)), None
                    except:
                        return None, "No embedded sequence"

    return None, "CDS not found"


# 🔥 SMART FETCH (skip useless fallback)
def fetch_cds(nuc_id, prot_id):
    seq, reason = fetch_via_protein(prot_id)

    if seq:
        return seq, None

    # 🚀 skip known useless cases
    if reason and ("400" in reason or "Empty" in reason):
        return None, "Skip fallback (bad protein)"

    tqdm.write(f"[fallback] {prot_id}")
    return fetch_via_nuccore(nuc_id, prot_id)


# ─────────────────────────── WORKER ───────────────────────────
def process_row(row, con):
    uid = row["UniProt_ID"]

    try:
        seq, reason = fetch_cds(
            row["GenBank_Nucleotide_ID"],
            row["GenBank_Protein_ID"]
        )
    except Exception as e:
        seq, reason = None, str(e)

    if seq:
        write_fasta(OUTPUT_FASTA, row, seq)
        mark_done(con, uid, "ok")
        return "ok"
    else:
        write_failed(FAILED_CSV, {
            "UniProt_ID": uid,
            "GenBank_Nucleotide_ID": row["GenBank_Nucleotide_ID"],
            "GenBank_Protein_ID": row["GenBank_Protein_ID"],
            "Reason": reason,
        })
        mark_done(con, uid, "failed")
        return "failed"


# ─────────────────────────── MAIN ───────────────────────────
def main():
    rows = list(csv.DictReader(open(INPUT_CSV)))

    print(f"Total rows: {len(rows):,}")

    con = init_db(PROGRESS_DB)
    done = load_done(con)

    todo = [r for r in rows if r["UniProt_ID"] not in done]

    print(f"Done: {len(done):,} | Remaining: {len(todo):,}\n")

    init_fasta(OUTPUT_FASTA)
    init_failed(FAILED_CSV)

    ok = fail = 0

    with ThreadPoolExecutor(max_workers=WORKERS) as ex:
        futures = [ex.submit(process_row, r, con) for r in todo]

        for f in tqdm(as_completed(futures), total=len(futures)):
            if f.result() == "ok":
                ok += 1
            else:
                fail += 1

    print(f"\nDone: {ok+fail:,}")
    print(f"Success: {ok:,}")
    print(f"Failed: {fail:,}")

    con.close()


if __name__ == "__main__":
    main()

In [ ]:
# creating isoform cds file

import gzip
import os
from typing import Iterator, Tuple, List, Optional
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import time
from datetime import datetime
import re

NCBI_EFETCH_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
UNIPROT_BASE_URL = "https://rest.uniprot.org/uniprotkb"
NCBI_RATE_LIMIT = 1.0

DEBUG_MODE = True
SAMPLE_SIZE = 15


class UniProtHeader:
#    """Parse UniProt FASTA header format."""
    PATTERN = re.compile(r'sp\|([A-Z0-9]+-\d+)\|')

    @staticmethod
    def extract_uniprot_id(header: str) -> Optional[str]:
        """Extract UniProt ID from FASTA header."""
        header = header.lstrip('>')
        match = UniProtHeader.PATTERN.search(header)
        if match:
            return match.group(1)
        return None


class UniProtFetcher:
#    """Fetch RefSeq NM IDs and Protein IDs from UniProt JSON."""

    def __init__(self):
        self.session = requests.Session()
        try:
            retry_strategy = Retry(
                total=3,
                status_forcelist=[429, 500, 502, 503, 504],
                allowed_methods=["GET"],
                backoff_factor=1
            )
        except TypeError:
            retry_strategy = Retry(
                total=3,
                status_forcelist=[429, 500, 502, 503, 504],
                method_whitelist=["GET"],
                backoff_factor=1
            )

        adapter = HTTPAdapter(max_retries=retry_strategy)
        self.session.mount("http://", adapter)
        self.session.mount("https://", adapter)

    def get_refseq_ids(self, uniprot_id: str, diagnostic: bool = False) -> Optional[Tuple[str, str]]:
#       """
#        Get RefSeq Protein ID (NP_) and Nucleotide ID (NM_) for a UniProt isoform ID.
#        Returns: (protein_id, nucleotide_id) or None
#      """
        try:
            url = f"{UNIPROT_BASE_URL}/{uniprot_id}.json"

            if diagnostic:
                print(f"  [DIAG] Fetching: {url}")

            response = self.session.get(url, timeout=10)

            if response.status_code != 200:
                if diagnostic:
                    print(f"  [DIAG] ❌ UniProt fetch failed with status {response.status_code}")
                return None

            if diagnostic:
                print(f"  [DIAG] ✅ UniProt response received (status 200)")

            data = response.json()

            if 'uniProtKBCrossReferences' not in data:
                if diagnostic:
                    print(f"  [DIAG] ❌ No 'uniProtKBCrossReferences' key in JSON response")
                return None

            xrefs = data['uniProtKBCrossReferences']

            if diagnostic:
                print(f"  [DIAG] ✅ Found {len(xrefs)} cross-references total")

            refseq_entries = [x for x in xrefs if x.get('database') == 'RefSeq']

            if diagnostic:
                print(f"  [DIAG] Found {len(refseq_entries)} RefSeq entries")

            if not refseq_entries:
                if diagnostic:
                    print(f"  [DIAG] ❌ No RefSeq database entries found")
                    databases = set(x.get('database') for x in xrefs)
                    print(f"  [DIAG] Available databases: {', '.join(sorted(databases))}")
                return None

            found_np_entries = []

            for idx, xref in enumerate(refseq_entries, 1):
                protein_id = xref.get('id', '')

                if diagnostic:
                    print(f"  [DIAG] Checking RefSeq entry #{idx}: {protein_id}")

                if not protein_id.startswith('NP_'):
                    if diagnostic:
                        print(f"  [DIAG]   ❌ Not NP_ format (starts with: {protein_id[:5]})")
                    continue

                found_np_entries.append(protein_id)

                nucleotide_id = None
                has_properties = 'properties' in xref

                if diagnostic:
                    print(f"  [DIAG]   ✅ NP_ format detected: {protein_id}")
                    print(f"  [DIAG]   Has properties: {has_properties}")

                if has_properties:
                    properties = xref['properties']
                    if diagnostic:
                        print(f"  [DIAG]   Properties count: {len(properties)}")

                    for prop_idx, prop in enumerate(properties, 1):
                        prop_key = prop.get('key', '')
                        prop_value = prop.get('value', '')

                        if diagnostic and prop_idx <= 5:
                            print(f"  [DIAG]     Property {prop_idx}: {prop_key} = {prop_value}")

                        if prop_key == 'NucleotideSequenceId':
                            nucleotide_id = prop_value
                            if diagnostic:
                                print(f"  [DIAG]     ✅ Found NucleotideSequenceId: {nucleotide_id}")
                            break

                if not nucleotide_id:
                    if diagnostic:
                        print(f"  [DIAG]   ❌ No NucleotideSequenceId property found")
                    continue

                if not nucleotide_id.startswith('NM_'):
                    if diagnostic:
                        print(f"  [DIAG]   ❌ NucleotideSequenceId not NM_ format: {nucleotide_id}")
                    continue

                if diagnostic:
                    print(f"  [DIAG] ✅ Found valid NP_/NM_ pair!")
                    print(f"  [DIAG]    Protein: {protein_id}")
                    print(f"  [DIAG]    Nucleotide: {nucleotide_id}")

                return (protein_id, nucleotide_id)

            if diagnostic:
                print(f"  [DIAG] ❌ No valid NP_/NM_ pair found")
                print(f"  [DIAG] Summary: RefSeq={len(refseq_entries)}, NP_={len(found_np_entries)}")

            return None

        except Exception as e:
            if diagnostic:
                print(f"  [DIAG] ❌ Exception: {type(e).__name__}: {e}")
            return None

    def close(self):
        self.session.close()


class NCBIFetcher:
#    """Fetch CDS sequences from NCBI using NM ID."""

    MAX_RETRIES = 3
    TIMEOUT = 30
    NCBI_EMAIL = "user@example.com"
    NCBI_API_KEY = None

    def __init__(self):
        self.session = requests.Session()
        self.last_request_time = 0

        try:
            retry_strategy = Retry(
                total=self.MAX_RETRIES,
                status_forcelist=[429, 500, 502, 503, 504],
                allowed_methods=["GET"],
                backoff_factor=2
            )
        except TypeError:
            retry_strategy = Retry(
                total=self.MAX_RETRIES,
                status_forcelist=[429, 500, 502, 503, 504],
                method_whitelist=["GET"],
                backoff_factor=2
            )

        adapter = HTTPAdapter(max_retries=retry_strategy)
        self.session.mount("http://", adapter)
        self.session.mount("https://", adapter)

    def _rate_limit(self):
        elapsed = time.time() - self.last_request_time
        if elapsed < NCBI_RATE_LIMIT:
            time.sleep(NCBI_RATE_LIMIT - elapsed)
        self.last_request_time = time.time()

    def fetch_cds_by_nm_id(self, nm_id: str, uniprot_id: str = "", diagnostic: bool = False) -> Optional[str]:
        try:
            self._rate_limit()

            params = {
                'db': 'nuccore',
                'id': nm_id,
                'rettype': 'gb',
                'retmode': 'text',
                'email': self.NCBI_EMAIL
            }

            if self.NCBI_API_KEY:
                params['api_key'] = self.NCBI_API_KEY

            if diagnostic:
                print(f"  [DIAG] Fetching from NCBI: NM ID = {nm_id}")

            response = self.session.get(NCBI_EFETCH_URL, params=params, timeout=self.TIMEOUT)

            if diagnostic:
                print(f"  [DIAG] NCBI response status: {response.status_code}")

            if response.status_code in [404, 400]:
                if diagnostic:
                    print(f"  [DIAG] ❌ NCBI returned 404/400 - NM ID not found or deprecated")
                return None

            if response.status_code != 200:
                if diagnostic:
                    print(f"  [DIAG] ❌ NCBI returned unexpected status: {response.status_code}")
                return None

            response.raise_for_status()

            gb_text = response.text

            if not gb_text or gb_text.startswith("Error"):
                if diagnostic:
                    print(f"  [DIAG] ❌ Empty or error response from NCBI")
                return None

            if diagnostic:
                print(f"  [DIAG] ✅ GenBank record received ({len(gb_text)} bytes)")

            cds_sequence = self._extract_first_cds_from_genbank(gb_text, uniprot_id, diagnostic)

            return cds_sequence

        except Exception as e:
            if diagnostic:
                print(f"  [DIAG] ❌ Exception: {e}")
            return None

    def _extract_first_cds_from_genbank(self, gb_text: str, uniprot_id: str = "", diagnostic: bool = False) -> Optional[str]:
        try:
            lines = gb_text.split('\n')

            sequence_start = None
            for i, line in enumerate(lines):
                if line.startswith('ORIGIN'):
                    sequence_start = i + 1
                    break

            if sequence_start is None:
                if diagnostic:
                    print(f"  [DIAG] ❌ ORIGIN section not found")
                return None

            if diagnostic:
                print(f"  [DIAG] ✅ ORIGIN section found")

            full_sequence = ''
            for i in range(sequence_start, len(lines)):
                line = lines[i]
                if line.startswith('//'):
                    break
                parts = line.split()
                for part in parts[1:]:
                    full_sequence += part

            if not full_sequence:
                if diagnostic:
                    print(f"  [DIAG] ❌ No sequence extracted from ORIGIN")
                return None

            if diagnostic:
                print(f"  [DIAG] ✅ Full sequence extracted: {len(full_sequence)} bp")

            cds_count = 0
            for i, line in enumerate(lines):
                if 'CDS' in line and line.startswith('     CDS'):
                    cds_count += 1
                    coords = line.split('CDS')[1].strip()

                    if diagnostic:
                        print(f"  [DIAG] CDS #{cds_count} found: {coords[:60]}")

                    try:
                        cds_seq = self._extract_cds_from_coords(coords, full_sequence)
                        if cds_seq and len(cds_seq) > 0:
                            if diagnostic:
                                print(f"  [DIAG] ✅ Extracted CDS: {len(cds_seq)} bp")
                            return cds_seq
                        else:
                            if diagnostic:
                                print(f"  [DIAG] ⚠️  CDS coordinate parsing failed, trying next CDS...")
                    except Exception as e:
                        if diagnostic:
                            print(f"  [DIAG] ⚠️  Failed to extract CDS #{cds_count}: {e}")
                        continue

            if diagnostic:
                print(f"  [DIAG] ❌ No valid CDS extracted (found {cds_count} CDS features total)")
            return None

        except Exception as e:
            if diagnostic:
                print(f"  [DIAG] ❌ Exception: {e}")
            return None

    def _extract_cds_from_coords(self, coords: str, full_sequence: str) -> Optional[str]:
        try:
            coords = coords.strip()
            is_complement = 'complement' in coords

            coords = coords.replace('complement(', '').replace('join(', '').replace(')', '')
            ranges = coords.split(',')

            cds_parts = []
            for range_str in ranges:
                range_str = range_str.strip()
                if '..' in range_str:
                    start_str, end_str = range_str.split('..')
                    start = int(start_str) - 1
                    end = int(end_str)
                    cds_parts.append(full_sequence[start:end])

            if not cds_parts:
                return None

            cds_seq = ''.join(cds_parts).upper()

            if is_complement:
                cds_seq = self._reverse_complement(cds_seq)

            return cds_seq if cds_seq else None

        except Exception:
            return None

    def _reverse_complement(self, seq: str) -> str:
        complement = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G', 'N': 'N'}
        return ''.join(complement.get(base, 'N') for base in reversed(seq))

    def close(self):
        self.session.close()


def read_fasta_from_file(file_path: str) -> Iterator[Tuple[str, str]]:
    if not os.path.exists(file_path):
        print(f"❌ ERROR: File not found: {file_path}")
        return

    with open(file_path, 'r', encoding='utf-8') as f:
        header = None
        seq_lines = []

        for line in f:
            line = line.rstrip('\n')

            if line.startswith('>'):
                if header is not None:
                    sequence = ''.join(seq_lines)
                    yield header, sequence

                header = line
                seq_lines = []
            else:
                seq_lines.append(line)

        if header is not None:
            sequence = ''.join(seq_lines)
            yield header, sequence


def extract_cds_from_fasta(input_fasta: str,
                          output_cds: str = 'output_cds.fasta.gz',
                          error_log: str = 'errors.log'):
#    """
#    Main function to extract CDS from FASTA file of isoforms.
#    With comprehensive diagnostics and final reporting.
#   """
    print(f"\n{'='*80}")
    print(f"🚀 CDS EXTRACTION FROM FASTA")
    print(f"{'='*80}")
    print(f"Input: {input_fasta}")
    print(f"Output: {output_cds}")
    print(f"Error log: {error_log}")
    print(f"Strategy: UniProt → RefSeq IDs → NCBI GenBank → Extract CDS")
    if DEBUG_MODE:
        print(f"Debug mode: ON (diagnostics for first {SAMPLE_SIZE} entries)")
    print(f"{'='*80}\n")

    if not os.path.exists(input_fasta):
        print(f"❌ ERROR: Input file not found: {input_fasta}")
        return

    start_time = datetime.now()

    uniprot_fetcher = UniProtFetcher()
    ncbi_fetcher = NCBIFetcher()

    total_processed = 0
    total_success = 0
    diagnostic_count = 0
    stats_summary = {
        'no_uniprot_id': 0,
        'no_refseq_ids': 0,
        'fetch_failed': 0,
        'success': 0
    }

    output_file = gzip.open(output_cds, 'wt', compresslevel=9, encoding='utf-8')
    error_file = open(error_log, 'w')
    error_file.write("uniprot_id\tprotein_id\tnm_id\terror_type\ttimestamp\n")

    try:
        for header, protein_seq in read_fasta_from_file(input_fasta):
            total_processed += 1

            uniprot_id = UniProtHeader.extract_uniprot_id(header)

            if not uniprot_id:
                stats_summary['no_uniprot_id'] += 1
                error_file.write(f"unknown\t\t\tno_uniprot_id\t{datetime.now().isoformat()}\n")
                if total_processed <= 5:
                    print(f"[{total_processed}] ❌ Could not extract UniProt ID")
                continue

            print(f"[{total_processed}] {uniprot_id}...", end=" ", flush=True)

            show_diagnostic = DEBUG_MODE and diagnostic_count < SAMPLE_SIZE

            refseq_ids = uniprot_fetcher.get_refseq_ids(uniprot_id, diagnostic=show_diagnostic)

            if not refseq_ids:
                diagnostic_count += 1
                stats_summary['no_refseq_ids'] += 1
                error_file.write(f"{uniprot_id}\t\t\tno_refseq_ids\t{datetime.now().isoformat()}\n")

                if show_diagnostic:
                    print(f"\n  Result: ❌ No RefSeq IDs found")
                else:
                    print(f"❌ No RefSeq IDs")

                if diagnostic_count == SAMPLE_SIZE + 1:
                    print(f"\n... (diagnostics complete, continuing silently) ...\n")

                continue

            protein_id, nm_id = refseq_ids
            print(f"NP/NM...", end=" ", flush=True)

            cds_seq = ncbi_fetcher.fetch_cds_by_nm_id(nm_id, uniprot_id, diagnostic=False)

            if cds_seq is None or len(cds_seq) == 0:
                stats_summary['fetch_failed'] += 1
                error_file.write(f"{uniprot_id}\t{protein_id}\t{nm_id}\tfetch_failed\t{datetime.now().isoformat()}\n")
                print(f"❌ Fetch failed")
                continue

            output_header = f"{uniprot_id}|{protein_id}|{nm_id}|{len(cds_seq)}bp"
            output_file.write(f'>{output_header}\n')

            for i in range(0, len(cds_seq), 80):
                output_file.write(f'{cds_seq[i:i+80]}\n')

            stats_summary['success'] += 1
            total_success += 1
            print(f"✅ {len(cds_seq)}bp")

            if total_processed % 100 == 0:
                elapsed_so_far = datetime.now() - start_time
                rate = total_processed / elapsed_so_far.total_seconds() if elapsed_so_far.total_seconds() > 0 else 0
                print(f"\n📊 Progress: {total_processed} processed, {total_success} successful ({rate:.2f} seqs/sec)\n")

    except KeyboardInterrupt:
        print("\n⚠️  Processing interrupted by user")
    except Exception as e:
        print(f"\n❌ Unexpected error: {e}")
        import traceback
        traceback.print_exc()

    finally:
        output_file.close()
        error_file.close()
        uniprot_fetcher.close()
        ncbi_fetcher.close()

    elapsed = datetime.now() - start_time

    # Final comprehensive report
    print(f"\n{'='*80}")
    print(f"✅ EXTRACTION COMPLETE!")
    print(f"{'='*80}")
    print(f"\n📊 COMPREHENSIVE SUMMARY:\n")
    print(f"Total processed:       {total_processed:,}")
    print(f"Successfully extracted: {stats_summary['success']:,} ({100*stats_summary['success']/max(total_processed,1):.1f}%)")
    print(f"\n❌ FAILURES:\n")
    print(f"  • No RefSeq IDs:       {stats_summary['no_refseq_ids']:,} ({100*stats_summary['no_refseq_ids']/max(total_processed,1):.1f}%)")
    print(f"    Reason: UniProt has no RefSeq link (protein may be non-human,")
    print(f"            old isoform, or lacks genomic sequence)")
    print(f"\n  • Fetch failed:        {stats_summary['fetch_failed']:,} ({100*stats_summary['fetch_failed']/max(total_processed,1):.1f}%)")
    print(f"    Reason: NM ID not found in NCBI or record has no CDS")
    print(f"\n  • No UniProt ID:       {stats_summary['no_uniprot_id']:,} ({100*stats_summary['no_uniprot_id']/max(total_processed,1):.1f}%)")
    print(f"    Reason: Header parsing failed")
    print(f"\n⏱️  Time elapsed: {elapsed}")
    print(f"\n📁 OUTPUT:\n")
    print(f"  • CDS FASTA: {output_cds}")
    print(f"  • Error log: {error_log}")
    print(f"\n{'='*80}\n")


# ============================================================================
# DIRECT EXECUTION
# ============================================================================

print("📦 Installing dependencies...")
os.system("pip install requests -q")
print("✅ Dependencies installed\n")

input_path = 'isoforms.fasta'

extract_cds_from_fasta(
    input_fasta=input_path,
    output_cds='isofom_cds.fasta.gz',
    error_log='errors.log'
)

In [ ]:
# dealing with isoform fail ids

import csv
import requests
import re
import os
from datetime import datetime
from tqdm import tqdm

# --------------------------------------------------
# CONFIG
# --------------------------------------------------
ERROR_LOG_FILE = "errors.log"
OUTPUT_RECOVERED = "recovered_iso.fasta"
RECOVERY_LOG = "recovery_iso.log"
SHOW_FIRST_N = 100

# --------------------------------------------------
# READ ERROR LOG
# --------------------------------------------------
def read_error_log(error_log_path):
    """Read error.log and extract no_refseq_ids entries."""
    entries = []

    if not os.path.exists(error_log_path):
        print(f"❌ Error log not found: {error_log_path}")
        return []

    with open(error_log_path, 'r') as f:
        reader = csv.DictReader(f, delimiter='\t')
        for row in reader:
            if row.get('error_type') == 'no_refseq_ids':
                uniprot_id = row.get('uniprot_id', '').strip()
                if uniprot_id:
                    entries.append(uniprot_id)

    return entries

# ==================================================
# STEP 1: FETCH UNIPROT TO GET UNIPARC ID
# ==================================================
def get_uniparc_id(uniprot_id):
#    """Fetch UniProt JSON and extract UniParc ID from extraAttributes."""
    try:
        url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.json"
        response = requests.get(url, timeout=10)

        if response.status_code != 200:
            return None

        data = response.json()

        # UniParc ID is in extraAttributes
        if 'extraAttributes' in data:
            uniparc_id = data['extraAttributes'].get('uniParcId')
            if uniparc_id:
                return uniparc_id

        # Fallback to top level
        return data.get('uniParcId')

    except Exception as e:
        return None

# ==================================================
# STEP 2: FETCH UNIPARC TO GET REFSEQ, EMBL & ENSEMBL IDS
# ==================================================
def get_all_ids_from_uniparc(uniparc_id):
#    """
#    Fetch UniParc JSON and extract:
#    - NP IDs (RefSeq protein)
#    - NM IDs (RefSeq mRNA)
#    - EMBL IDs (European Nucleotide Archive)
#    - Ensembl IDs (Ensembl protein)
#    """
    results = {
        'np_id': None,
        'nm_id': None,
        'embl_id': None,
        'ensembl_protein_id': None
    }

    try:
        url = f"https://rest.uniprot.org/uniparc/{uniparc_id}.json"
        response = requests.get(url, timeout=10)

        if response.status_code != 200:
            return results

        data = response.json()

        # Loop through cross-references
        if 'uniParcCrossReferences' in data:
            for xref in data['uniParcCrossReferences']:
                database = xref.get('database', '')
                ref_id = xref.get('id', '')
                active = xref.get('active', False)

                # Extract RefSeq IDs
                if database == 'RefSeq':
                    if ref_id.startswith('NM_'):
                        results['nm_id'] = ref_id
                    elif ref_id.startswith('NP_'):
                        results['np_id'] = ref_id

                # Extract EMBL IDs (prefer active)
                if database == 'EMBL' and active and not results['embl_id']:
                    results['embl_id'] = ref_id

                # Extract Ensembl protein IDs (prefer active)
                if database == 'Ensembl' and active and not results['ensembl_protein_id']:
                    results['ensembl_protein_id'] = ref_id

        return results

    except Exception as e:
        return results

# ==================================================
# PIPELINE 1: NM EXTRACTION
# ==================================================

def extract_nm_from_np_genbank(np_id):
#    """
#    Fetch NP GenBank record and extract NM ID from DBSOURCE line.
#    Format: DBSOURCE    REFSEQ: accession NM_XXXXX.X
#    """
    try:
        url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
        params = {
            'db': 'protein',
            'id': np_id,
            'rettype': 'gb',
            'retmode': 'text',
            'email': 'user@example.com'
        }

        response = requests.get(url, params=params, timeout=15)

        if response.status_code != 200:
            return None

        gb_text = response.text

        # Search for DBSOURCE line
        nm_match = re.search(r'DBSOURCE\s+REFSEQ:\s+accession\s+(NM_\d+\.\d+)', gb_text)

        if nm_match:
            return nm_match.group(1)

        return None

    except Exception as e:
        return None


def fetch_cds_by_nm(nm_id):
#    """Fetch CDS sequence from NM ID using GenBank format."""
    try:
        params = {
            'db': 'nuccore',
            'id': nm_id,
            'rettype': 'gb',
            'retmode': 'text',
            'email': 'user@example.com'
        }

        url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
        response = requests.get(url, params=params, timeout=15)

        if response.status_code != 200:
            return None

        gb_text = response.text
        cds_seq = extract_cds_from_genbank(gb_text)

        return cds_seq

    except Exception as e:
        return None

# ==================================================
# PIPELINE 2: EMBL EXTRACTION (WITH FALLBACK)
# ==================================================

def fetch_cds_from_embl(embl_id):
#    """Fetch CDS sequence from EMBL database via EBI API."""
    try:
        url = f"https://www.ebi.ac.uk/ena/browser/api/embl/{embl_id}"
        response = requests.get(url, timeout=15)

        if response.status_code != 200:
            return None

        embl_text = response.text
        cds_seq = extract_cds_from_embl_record(embl_text)

        return cds_seq

    except Exception as e:
        return None


def extract_cds_from_embl_record(embl_text):
#    """
#    Extract CDS from EMBL/ENA flat file format.
#
#    Handles:
#   1. Direct CDS coordinates: "FT   CDS             20..613"
#    2. CDS references: "FT   CDS             BC014484.1:20..613"
#    3. Multi-line coordinates
#    4. Complement sequences
#    5. FALLBACK: Returns full sequence if CDS extraction fails
#    """
    try:
        lines = embl_text.split('\n')

        # Find SQ (sequence) section
        sequence_start = None
        for i, line in enumerate(lines):
            if line.startswith('SQ '):
                sequence_start = i + 1
                break

        if sequence_start is None:
            return None

        # Extract sequence from SQ section
        full_sequence = ''
        for i in range(sequence_start, len(lines)):
            line = lines[i]

            if line.startswith('//'):
                break

            # Remove all non-ACGT characters
            seq_part = re.sub(r'[^ACGTacgt]', '', line)
            full_sequence += seq_part.upper()

        if not full_sequence:
            return None

        # Try to extract CDS using coordinates
        cds_seq = None

        # Find CDS feature
        for i, line in enumerate(lines):
            if line.startswith('FT   CDS'):
                coords_str = line.replace('FT   CDS', '').strip()

                # ✅ Collect continuation lines until we hit a qualifier (/)
                j = i + 1
                while j < len(lines) and lines[j].startswith('FT                   '):
                    next_line = lines[j].replace('FT                   ', '').strip()

                    # Stop at qualifiers
                    if next_line.startswith('/'):
                        break

                    coords_str += ' ' + next_line
                    j += 1

                coords_str = coords_str.strip()

                # ✅ Check if CDS references another sequence
                referenced_accession = None
                if ':' in coords_str:
                    accession_ref = coords_str.split(':')[0]
                    coords_str = coords_str.split(':')[-1]
                    referenced_accession = accession_ref

                    # Check if referenced accession is in current record
                    has_this_accession = False
                    for line in lines:
                        if line.startswith('PA   ') and accession_ref in line:
                            has_this_accession = True
                            break

                    # If referenced sequence is different, try to fetch it
                    if not has_this_accession:
                        alt_cds = fetch_cds_from_referenced_accession(accession_ref, coords_str)
                        if alt_cds and len(alt_cds) > 0:
                            return alt_cds
                        # If fetching fails, continue to fallback

                # ✅ Try to extract CDS using coordinates
                try:
                    # Handle complement
                    is_complement = 'complement' in coords_str
                    coords_str = coords_str.replace('complement(', '').replace('join(', '').replace(')', '')

                    # Split by comma for multiple ranges
                    ranges = coords_str.split(',')

                    cds_parts = []
                    for range_str in ranges:
                        range_str = range_str.strip()

                        if '..' in range_str:
                            start_str, end_str = range_str.split('..')
                            start = int(start_str.strip()) - 1  # 0-based
                            end = int(end_str.strip())

                            # ✅ Check if coordinates are within bounds
                            if 0 <= start < len(full_sequence) and 0 < end <= len(full_sequence):
                                cds_parts.append(full_sequence[start:end])

                    if cds_parts:
                        cds_seq = ''.join(cds_parts)

                        # Apply reverse complement if needed
                        if is_complement:
                            complement = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G', 'N': 'N'}
                            cds_seq = ''.join(complement.get(base, 'N') for base in reversed(cds_seq))

                        if cds_seq and len(cds_seq) > 0:
                            return cds_seq

                except (ValueError, IndexError):
                    # Coordinates parsing failed, continue to fallback
                    pass

                # Break after first CDS attempt (don't try other CDS features)
                break

        # ✅ FALLBACK: If CDS extraction failed or coordinates were invalid,
        # return the full sequence as-is
        if full_sequence and len(full_sequence) > 0:
            return full_sequence

        return None

    except Exception as e:
        return None


def fetch_cds_from_referenced_accession(accession, coords):
#   """
#    Fetch CDS from a referenced accession in ENA.
#   Used when CDS is defined in a different sequence record.
#
#    Example: CDS points to "BC014484.1:20..613"
#    Try EBI first, then NCBI as fallback.
#    """
    try:
        # Try EBI ENA API first
        url = f"https://www.ebi.ac.uk/ena/browser/api/embl/{accession}"
        response = requests.get(url, timeout=15)

        if response.status_code == 200:
            embl_text = response.text
            return extract_cds_from_referenced_embl(embl_text, coords)

        # Fallback to NCBI
        return fetch_cds_from_ncbi_nucleotide(accession, coords)

    except Exception as e:
        return None


def extract_cds_from_referenced_embl(embl_text, coords):
#    """Extract CDS from a referenced EMBL record using provided coordinates."""
    try:
        lines = embl_text.split('\n')

        # Find SQ section
        sequence_start = None
        for i, line in enumerate(lines):
            if line.startswith('SQ '):
                sequence_start = i + 1
                break

        if sequence_start is None:
            return None

        # Extract full sequence
        full_sequence = ''
        for i in range(sequence_start, len(lines)):
            line = lines[i]

            if line.startswith('//'):
                break

            seq_part = re.sub(r'[^ACGTacgt]', '', line)
            full_sequence += seq_part.upper()

        if not full_sequence:
            return None

        # Extract CDS using provided coordinates
        cds_seq = extract_cds_from_coords(coords, full_sequence)

        return cds_seq

    except Exception as e:
        return None


def fetch_cds_from_ncbi_nucleotide(accession, coords):
#    """
#    Fetch CDS from NCBI nucleotide database.
#    Used as fallback when EBI is unavailable.
#    """
    try:
        params = {
            'db': 'nuccore',
            'id': accession,
            'rettype': 'gb',
            'retmode': 'text',
            'email': 'user@example.com'
        }

        url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
        response = requests.get(url, params=params, timeout=15)

        if response.status_code != 200:
            return None

        gb_text = response.text

        # Extract full sequence from GenBank
        lines = gb_text.split('\n')

        sequence_start = None
        for i, line in enumerate(lines):
            if line.startswith('ORIGIN'):
                sequence_start = i + 1
                break

        if sequence_start is None:
            return None

        full_sequence = ''
        for i in range(sequence_start, len(lines)):
            line = lines[i]
            if line.startswith('//'):
                break
            parts = line.split()
            for part in parts[1:]:
                full_sequence += part

        if not full_sequence:
            return None

        # Extract CDS using provided coordinates
        cds_seq = extract_cds_from_coords(coords, full_sequence)

        return cds_seq

    except Exception as e:
        return None

# ==================================================
# PIPELINE 3: ENSEMBL EXTRACTION (NEW!)
# ==================================================

def fetch_cds_from_ensembl(ensembl_protein_id):
#    """
#   Fetch CDS sequence from Ensembl.
#
#    Strategy:
#    1. Lookup protein ID to get transcript ID
#    2. Fetch CDS from transcript ID using type=cds
#    """
    try:
        # Step 1: Get transcript ID
        transcript_id = lookup_ensembl_transcript(ensembl_protein_id)

        if not transcript_id:
            return None

        # Step 2: Fetch CDS from transcript
        cds_seq = fetch_ensembl_cds_from_transcript(transcript_id)

        return cds_seq

    except Exception as e:
        return None


def lookup_ensembl_transcript(ensembl_protein_id):
#    """
#    Lookup Ensembl transcript ID from protein ID.
#    Uses Ensembl REST API lookup endpoint.
#    """
    try:
        url = f"https://rest.ensembl.org/lookup/id/{ensembl_protein_id}"
        headers = {'Content-Type': 'application/json'}

        response = requests.get(url, headers=headers, timeout=15)

        if response.status_code != 200:
            return None

        data = response.json()

        # Get transcript ID (stored in 'Parent' field)
        transcript_id = data.get('Parent')

        return transcript_id

    except Exception as e:
        return None


def fetch_ensembl_cds_from_transcript(ensembl_transcript_id):
#    """
#    Fetch CDS sequence from Ensembl transcript ID.
#    Returns nucleotide CDS sequence (A, T, G, C).
#    """
    try:
        url = f"https://rest.ensembl.org/sequence/id/{ensembl_transcript_id}"

        params = {
            'type': 'cds'  # ✅ CRITICAL: Must request CDS, not protein
        }

        headers = {'Content-Type': 'application/json'}

        response = requests.get(url, params=params, headers=headers, timeout=15)

        if response.status_code != 200:
            return None

        data = response.json()

        # Get sequence
        seq = data.get('seq')

        if seq:
            # Ensure it's nucleotide sequence (should be A, T, G, C)
            seq = seq.upper()

            # Validate it's DNA (not protein)
            if all(base in 'ATGC' for base in seq):
                return seq
            else:
                # Got protein sequence, not CDS
                return None

        return None

    except Exception as e:
        return None

# ==================================================
# HELPER FUNCTIONS
# ==================================================

def extract_cds_from_genbank(gb_text):
#    """Extract first CDS from GenBank text."""
    try:
        lines = gb_text.split('\n')

        # Find ORIGIN section
        sequence_start = None
        for i, line in enumerate(lines):
            if line.startswith('ORIGIN'):
                sequence_start = i + 1
                break

        if sequence_start is None:
            return None

        # Extract sequence
        full_sequence = ''
        for i in range(sequence_start, len(lines)):
            line = lines[i]
            if line.startswith('//'):
                break
            parts = line.split()
            for part in parts[1:]:
                full_sequence += part

        if not full_sequence:
            return None

        # Find first CDS
        for i, line in enumerate(lines):
            if 'CDS' in line and line.startswith('     CDS'):
                coords = line.split('CDS')[1].strip()

                try:
                    cds_seq = extract_cds_from_coords(coords, full_sequence)
                    if cds_seq and len(cds_seq) > 0:
                        return cds_seq
                except:
                    continue

        return None

    except Exception as e:
        return None


def extract_cds_from_coords(coords, full_sequence):
#    """
#    Extract CDS from coordinates string.
#
#    Handles:
#    - Simple ranges: "1..195"
#    - Multiple ranges: "join(1..50,100..200)"
#    - Complement: "complement(1..195)"
#    - Combined: "complement(join(...))"
#    """
    try:
        coords = coords.strip()
        is_complement = 'complement' in coords

        # Remove modifiers
        coords = coords.replace('complement(', '').replace('join(', '').replace(')', '')
        ranges = coords.split(',')

        cds_parts = []
        for range_str in ranges:
            range_str = range_str.strip()
            if '..' in range_str:
                start_str, end_str = range_str.split('..')
                start = int(start_str.strip()) - 1  # Convert to 0-based
                end = int(end_str.strip())

                if 0 <= start < len(full_sequence) and 0 < end <= len(full_sequence):
                    cds_parts.append(full_sequence[start:end])

        if not cds_parts:
            return None

        cds_seq = ''.join(cds_parts).upper()

        # Apply reverse complement if needed
        if is_complement:
            cds_seq = reverse_complement(cds_seq)

        return cds_seq if cds_seq else None

    except Exception as e:
        return None


def reverse_complement(seq):
#    """Get reverse complement of DNA sequence."""
    complement = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G', 'N': 'N'}
    return ''.join(complement.get(base, 'N') for base in reversed(seq))

# ==================================================
# MAIN RECOVERY PIPELINE
# ==================================================

def recovery_pipeline():
#    """
#    Main recovery pipeline with three strategies:
#    1. NM extraction from RefSeq
#    2. EMBL database with reference handling and full sequence fallback
#    3. Ensembl protein→transcript→CDS lookup
#    """

    print(f"\n{'='*80}")
    print(f"🚀 RECOVERY PIPELINE: UniParcID → RefSeq/EMBL/Ensembl → CDS")
    print(f"{'='*80}")
    print(f"   Pipeline 1: Extract NM from RefSeq")
    print(f"   Pipeline 2: EMBL database (with full sequence fallback)")
    print(f"   Pipeline 3: Ensembl protein→transcript→CDS (NEW!)")
    print(f"{'='*80}\n")

    # Read error log
    print(f"📖 Reading error log...")
    failed_ids = read_error_log(ERROR_LOG_FILE)
    print(f"✅ Found {len(failed_ids):,} entries with no_refseq_ids\n")

    if not failed_ids:
        print("❌ No entries to process!")
        return False

    # Initialize statistics
    stats = {
        'total_attempted': len(failed_ids),
        'uniparc_found': 0,
        'np_found': 0,
        'nm_found_direct': 0,
        'nm_extracted_from_np': 0,
        'embl_found': 0,
        'ensembl_found': 0,
        'nm_cds_success': 0,
        'embl_cds_success': 0,
        'ensembl_cds_success': 0,
        'total_recovered': 0,
        'failed': 0
    }

    print(f"\n{'='*80}")
    print(f"📋 FIRST {SHOW_FIRST_N} ENTRIES - DETAILED OUTPUT")
    print(f"{'='*80}\n")

    with open(OUTPUT_RECOVERED, 'w') as out_f, \
         open(RECOVERY_LOG, 'w') as log_f:

        log_f.write("uniprot_id\tuniparc_id\tnp_id\tnm_id\tembl_id\tensembl_id\tstrategy\tstatus\tcds_length\ttimestamp\n")

        for idx, uniprot_id in enumerate(tqdm(failed_ids, desc="Processing entries"), 1):

            show_details = idx <= SHOW_FIRST_N

            if show_details:
                print(f"\n[{idx}] {uniprot_id}")

            try:
                # ========== STEP 1: Get UniParc ID ==========
                if show_details:
                    print(f"  [Step 1] Fetching UniParc ID...", end=" ", flush=True)

                uniparc_id = get_uniparc_id(uniprot_id)

                if not uniparc_id:
                    if show_details:
                        print(f"❌ FAILED")
                    stats['failed'] += 1
                    log_f.write(f"{uniprot_id}\tN/A\tN/A\tN/A\tN/A\tN/A\tN/A\tFAILED_NO_UNIPARC\t0\t{datetime.now().isoformat()}\n")
                    continue

                if show_details:
                    print(f"✅ {uniparc_id}")

                stats['uniparc_found'] += 1

                # ========== STEP 2: Get RefSeq, EMBL & Ensembl IDs ==========
                if show_details:
                    print(f"  [Step 2] Fetching RefSeq/EMBL/Ensembl IDs...", end=" ", flush=True)

                cross_refs = get_all_ids_from_uniparc(uniparc_id)
                np_id = cross_refs['np_id']
                nm_id = cross_refs['nm_id']
                embl_id = cross_refs['embl_id']
                ensembl_protein_id = cross_refs['ensembl_protein_id']

                if not np_id and not nm_id and not embl_id and not ensembl_protein_id:
                    if show_details:
                        print(f"❌ FAILED - No IDs found")
                    stats['failed'] += 1
                    log_f.write(f"{uniprot_id}\t{uniparc_id}\tN/A\tN/A\tN/A\tN/A\tN/A\tFAILED_NO_IDS\t0\t{datetime.now().isoformat()}\n")
                    continue

                if show_details:
                    ids_str = ""
                    if nm_id:
                        ids_str += f"NM: {nm_id}"
                    if np_id:
                        if ids_str:
                            ids_str += ", "
                        ids_str += f"NP: {np_id}"
                    if embl_id:
                        if ids_str:
                            ids_str += ", "
                        ids_str += f"EMBL: {embl_id}"
                    if ensembl_protein_id:
                        if ids_str:
                            ids_str += ", "
                        ids_str += f"Ensembl: {ensembl_protein_id}"
                    print(f"✅ {ids_str}")

                if np_id:
                    stats['np_found'] += 1
                if nm_id:
                    stats['nm_found_direct'] += 1
                if embl_id:
                    stats['embl_found'] += 1
                if ensembl_protein_id:
                    stats['ensembl_found'] += 1

                cds_seq = None
                strategy_used = None

                # ========== PIPELINE 1: Direct NM ==========
                if nm_id and not cds_seq:
                    if show_details:
                        print(f"  [P1-S1] Fetching CDS from NM (direct): {nm_id}...", end=" ", flush=True)

                    cds_seq = fetch_cds_by_nm(nm_id)
                    if cds_seq and len(cds_seq) > 0:
                        if show_details:
                            print(f"✅ {len(cds_seq)} bp")
                        stats['nm_cds_success'] += 1
                        strategy_used = "NM_Direct"
                    else:
                        if show_details:
                            print(f"❌ Failed")

                # ========== PIPELINE 1: Extract NM from NP ==========
                if not cds_seq and np_id:
                    if show_details:
                        print(f"  [P1-S2] Extracting NM from NP: {np_id}...", end=" ", flush=True)

                    extracted_nm_id = extract_nm_from_np_genbank(np_id)

                    if extracted_nm_id:
                        if show_details:
                            print(f"✅ {extracted_nm_id}")
                            print(f"  [P1-S3] Fetching CDS from extracted NM...", end=" ", flush=True)

                        stats['nm_extracted_from_np'] += 1
                        nm_id = extracted_nm_id

                        cds_seq = fetch_cds_by_nm(extracted_nm_id)

                        if cds_seq and len(cds_seq) > 0:
                            if show_details:
                                print(f"✅ {len(cds_seq)} bp")
                            stats['nm_cds_success'] += 1
                            strategy_used = "NM_ExtractedFromNP"
                        else:
                            if show_details:
                                print(f"❌ Failed")
                    else:
                        if show_details:
                            print(f"❌ Could not extract NM from NP")

                # ========== PIPELINE 2: EMBL ==========
                if not cds_seq and embl_id:
                    if show_details:
                        print(f"  [P2-EMBL] Fetching CDS from EMBL: {embl_id}...", end=" ", flush=True)

                    cds_seq = fetch_cds_from_embl(embl_id)

                    if cds_seq and len(cds_seq) > 0:
                        if show_details:
                            print(f"✅ {len(cds_seq)} bp")
                        stats['embl_cds_success'] += 1
                        strategy_used = "EMBL"
                    else:
                        if show_details:
                            print(f"❌ Failed")

                # ========== PIPELINE 3: ENSEMBL (NEW!) ==========
                if not cds_seq and ensembl_protein_id:
                    if show_details:
                        print(f"  [P3-Ensembl] Fetching CDS from Ensembl: {ensembl_protein_id}...", end=" ", flush=True)

                    cds_seq = fetch_cds_from_ensembl(ensembl_protein_id)

                    if cds_seq and len(cds_seq) > 0:
                        if show_details:
                            print(f"✅ {len(cds_seq)} bp")
                        stats['ensembl_cds_success'] += 1
                        strategy_used = "Ensembl"
                    else:
                        if show_details:
                            print(f"❌ Failed")

                # ========== LOG RESULTS ==========
                if cds_seq and len(cds_seq) > 0:
                    stats['total_recovered'] += 1

                    if show_details:
                        print(f"  ✅ RECOVERED ({strategy_used}): {len(cds_seq)} bp")

                    # Write FASTA
                    out_f.write(f">{uniprot_id}|{np_id or 'N/A'}|{nm_id or 'N/A'}|{embl_id or 'N/A'}|{ensembl_protein_id or 'N/A'}|{len(cds_seq)}bp|{strategy_used}\n")
                    for i in range(0, len(cds_seq), 80):
                        out_f.write(f"{cds_seq[i:i+80]}\n")

                    log_f.write(f"{uniprot_id}\t{uniparc_id}\t{np_id or 'N/A'}\t{nm_id or 'N/A'}\t{embl_id or 'N/A'}\t{ensembl_protein_id or 'N/A'}\t{strategy_used}\tSUCCESS\t{len(cds_seq)}\t{datetime.now().isoformat()}\n")

                else:
                    stats['failed'] += 1
                    if show_details:
                        print(f"  ❌ FAILED - No CDS retrieved from any source")
                    log_f.write(f"{uniprot_id}\t{uniparc_id}\t{np_id or 'N/A'}\t{nm_id or 'N/A'}\t{embl_id or 'N/A'}\t{ensembl_protein_id or 'N/A'}\tN/A\tFAILED_NO_CDS\t0\t{datetime.now().isoformat()}\n")

            except Exception as e:
                stats['failed'] += 1
                if show_details:
                    print(f"  ❌ EXCEPTION: {str(e)[:50]}")
                log_f.write(f"{uniprot_id}\tERROR\tN/A\tN/A\tN/A\tN/A\tN/A\tFAILED_EXCEPTION\t0\t{datetime.now().isoformat()}\n")

    # ========== PRINT SUMMARY ==========
    print(f"\n\n{'='*80}")
    print(f"✅ RECOVERY COMPLETE")
    print(f"{'='*80}\n")
    print(f"📊 COMPREHENSIVE SUMMARY:\n")
    print(f"  Total attempted:                  {stats['total_attempted']:,}")
    print(f"\n  📍 Discovery Phase:")
    print(f"     UniParc IDs found:             {stats['uniparc_found']:,}")
    print(f"     NP IDs found:                  {stats['np_found']:,}")
    print(f"     NM IDs found (direct):         {stats['nm_found_direct']:,}")
    print(f"     NM IDs extracted from NP:      {stats['nm_extracted_from_np']:,}")
    print(f"     EMBL IDs found:                {stats['embl_found']:,}")
    print(f"     Ensembl IDs found:             {stats['ensembl_found']:,}")
    print(f"\n  🔬 Extraction Phase:")
    print(f"     NM → CDS success:              {stats['nm_cds_success']:,}")
    print(f"     EMBL → CDS:                    {stats['embl_cds_success']:,}")
    print(f"     Ensembl → CDS:                 {stats['ensembl_cds_success']:,}")
    print(f"\n  📈 Final Results:")
    print(f"     ✅ TOTAL RECOVERED:            {stats['total_recovered']:,} ({100*stats['total_recovered']/max(stats['total_attempted'],1):.1f}%)")
    print(f"     ❌ FAILED:                     {stats['failed']:,} ({100*stats['failed']/max(stats['total_attempted'],1):.1f}%)")
    print(f"\n💾 Output files:")
    print(f"  • CDS sequences: {OUTPUT_RECOVERED}")
    print(f"  • Recovery log:  {RECOVERY_LOG}")
    print(f"\n{'='*80}\n")

    return True

# ==================================================
# EXECUTION
# ==================================================

if __name__ == "__main__":
    print("📦 Installing dependencies...")
    os.system("pip install requests tqdm -q")
    print("✅ Dependencies installed\n")

    success = recovery_pipeline()

    if success:
        print("📥 Preparing to download results...")
    else:
        print("❌ Recovery pipeline failed!")


In [ ]:
import os
import gzip

# --------------------------------------------------
# CONFIG
# --------------------------------------------------
FILE1 = "isofom_cds.fasta.gz"
FILE2 = "recovered_iso.fasta"

OUTPUT_FILE = "iso_combined_cds.fasta"

# --------------------------------------------------
# OPEN FASTA OR FASTA.GZ
# --------------------------------------------------
def open_fasta(filepath):

    if filepath.endswith(".gz"):
        return gzip.open(filepath, "rt")

    return open(filepath, "r")


# --------------------------------------------------
# PRINT HEAD OF FILES
# --------------------------------------------------
def print_file_head(filepath, num_lines=10):

    print(f"\n{'='*80}")
    print(f"📄 HEAD of {os.path.basename(filepath)} (first {num_lines} lines)")
    print(f"{'='*80}\n")

    try:

        with open_fasta(filepath) as f:

            for i, line in enumerate(f):

                if i >= num_lines:
                    break

                print(line.rstrip())

        print("\n✅ File exists and readable")

    except FileNotFoundError:
        print(f"❌ File not found: {filepath}")

    except Exception as e:
        print(f"❌ Error: {e}")


# --------------------------------------------------
# GET FILE STATISTICS
# --------------------------------------------------
def get_file_stats(filepath):

    print(f"\n{'='*80}")
    print(f"📊 FILE STATISTICS: {os.path.basename(filepath)}")
    print(f"{'='*80}\n")

    try:

        num_sequences = 0
        total_bp = 0
        total_lines = 0

        with open_fasta(filepath) as f:

            for line in f:

                total_lines += 1

                if line.startswith(">"):
                    num_sequences += 1
                else:
                    total_bp += len(line.strip())

        print(f"  Total sequences:   {num_sequences:,}")
        print(f"  Total nucleotides: {total_bp:,} bp")

        if num_sequences > 0:
            print(f"  Average length:    {total_bp/num_sequences:.0f} bp/sequence")

        print(f"  Total lines:       {total_lines:,}")

        size = os.path.getsize(filepath)

        print(f"  File size:         {size/(1024*1024):.2f} MB")

        return {
            "sequences": num_sequences,
            "nucleotides": total_bp
        }

    except FileNotFoundError:
        print(f"❌ File not found: {filepath}")
        return None

    except Exception as e:
        print(f"❌ Error: {e}")
        return None


# --------------------------------------------------
# COUNT SEQUENCES
# --------------------------------------------------
def count_sequences(filepath):

    count = 0

    with open_fasta(filepath) as f:

        for line in f:

            if line.startswith(">"):
                count += 1

    return count


# --------------------------------------------------
# JOIN FASTA FILES
# --------------------------------------------------
def join_fasta_files(file1, file2, output_file):

    print(f"\n{'='*80}")
    print("🔗 JOINING FASTA FILES")
    print(f"{'='*80}\n")

    try:

        count1 = count_sequences(file1)
        print(f"📊 File 1: {count1:,} sequences")

        count2 = count_sequences(file2)
        print(f"📊 File 2: {count2:,} sequences")

        print("\n🔄 Combining files...")

        with open(output_file, "w") as outf:

            with open_fasta(file1) as f:

                for line in f:
                    outf.write(line)

            with open_fasta(file2) as f:

                for line in f:
                    outf.write(line)

        total = count1 + count2

        print("\n✅ Combined file created!")
        print(f"📊 Total sequences: {total:,}")
        print(f"💾 Output file: {output_file}")

        size = os.path.getsize(output_file)

        print(f"📦 File size: {size/(1024*1024):.2f} MB")

        return True

    except Exception as e:

        print(f"❌ Error: {e}")

        import traceback
        traceback.print_exc()

        return False


# --------------------------------------------------
# MAIN
# --------------------------------------------------
if __name__ == "__main__":

    print(f"\n{'='*80}")
    print("🚀 FASTA FILE COMBINER")
    print(f"{'='*80}\n")

    print_file_head(FILE1)
    print_file_head(FILE2)

    stats1 = get_file_stats(FILE1)
    stats2 = get_file_stats(FILE2)

    success = join_fasta_files(
        FILE1,
        FILE2,
        OUTPUT_FILE
    )

    if success:

        combined_stats = get_file_stats(
            OUTPUT_FILE
        )

        print(f"\n{'='*80}")
        print("✅ COMBINATION SUMMARY")
        print(f"{'='*80}\n")

        if stats1 and stats2 and combined_stats:

            print(f"File 1 sequences:   {stats1['sequences']:,}")
            print(f"File 2 sequences:   {stats2['sequences']:,}")
            print(f"Combined sequences: {combined_stats['sequences']:,}")

            if (
                combined_stats["sequences"]
                ==
                stats1["sequences"] + stats2["sequences"]
            ):
                print("\n✅ Verification PASSED")
            else:
                print("\n❌ Verification FAILED")

        print(f"\nCombined FASTA saved as:\n{OUTPUT_FILE}")

In [ ]:
from tqdm import tqdm

# ================= SETTINGS =================
FASTA_FILES = [
    "non_isoform_cds_output.fasta",
    "iso_combined_cds.fasta"
]

OUTPUT_FASTA = "uniprot_cds.fasta"
# ============================================

total_lines = 0

with open(OUTPUT_FASTA, "w") as outfile:

    for fasta_file in FASTA_FILES:

        print(f"\nMerging: {fasta_file}")

        with open(fasta_file, "r") as infile:

            for line in tqdm(infile, desc=fasta_file):

                outfile.write(line)
                total_lines += 1

print("\n===== DONE =====")
print(f"Merged {len(FASTA_FILES)} FASTA files")
print(f"Total lines written: {total_lines:,}")
print(f"Output saved to: {OUTPUT_FASTA}")

In [ ]:
# checking all pentapeptides in prot seq

from itertools import product
from collections import Counter
from tqdm import tqdm
import csv

# ---------- INPUT / OUTPUT ----------
fasta_file = "complete_uniprot_prot.fasta"

freq_output = "pentapeptide_frequencies.csv"
zero_output = "absent_pep.csv"

# Standard amino acids
AA = set("ACDEFGHIKLMNPQRSTVWY")


# ---------- STEP 0: COUNT PROTEINS ----------
print("Counting protein entries...")

header_count = 0

with open(fasta_file, "r", encoding="utf-8") as f:
    for line in f:
        if line.startswith(">"):
            header_count += 1

print(f"Total proteins found: {header_count}")


# ---------- STEP 1: COUNT PENTAPEPTIDES ----------
print("\nCounting pentapeptides...")

pent_counts = Counter()

current_seq = []

with open(fasta_file, "r", encoding="utf-8") as f:

    pbar = tqdm(total=header_count, desc="Processing proteins")

    for line in f:

        line = line.strip()

        # New protein
        if line.startswith(">"):

            # Process previous sequence
            if current_seq:

                seq = "".join(current_seq).upper()

                seq_len = len(seq)

                for i in range(seq_len - 4):

                    pep = seq[i:i+5]

                    # Faster amino acid validation
                    if set(pep) <= AA:
                        pent_counts[pep] += 1

            current_seq = []

            pbar.update(1)

        else:
            current_seq.append(line)

    # Process final sequence
    if current_seq:

        seq = "".join(current_seq).upper()

        seq_len = len(seq)

        for i in range(seq_len - 4):

            pep = seq[i:i+5]

            if set(pep) <= AA:
                pent_counts[pep] += 1

    pbar.close()


# ---------- STEP 2: WRITE OUTPUT ----------
print("\nWriting CSV files...")

total_possible = 20**5
zero_count = 0

with open(freq_output, "w", newline="", encoding="utf-8") as freq_out, \
     open(zero_output, "w", newline="", encoding="utf-8") as zero_out:

    freq_writer = csv.writer(freq_out)
    zero_writer = csv.writer(zero_out)

    freq_writer.writerow(["Pentapeptide", "Frequency"])
    zero_writer.writerow(["Pentapeptide"])

    for pep_tuple in tqdm(product("ACDEFGHIKLMNPQRSTVWY", repeat=5),
                          total=total_possible,
                          desc="Generating all pentapeptides"):

        pep = "".join(pep_tuple)

        freq = pent_counts.get(pep, 0)

        freq_writer.writerow([pep, freq])

        if freq == 0 and "CQWW" not in pep:
            zero_writer.writerow([pep])
            zero_count += 1


# ---------- FINAL STATS ----------
print("\nDone!")

print(f"Observed unique pentapeptides: {len(pent_counts)}")
print(f"Total possible pentapeptides: {total_possible}")
print(f"Zero-frequency pentapeptides: {zero_count}")

print(f"\nFrequency CSV: {freq_output}")
print(f"Zero-frequency CSV: {zero_output}")

In [ ]:
# adding translation table and codon start info to cds fasta

from Bio import Entrez, SeqIO
from tqdm import tqdm
import time
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from Bio import BiopythonParserWarning

warnings.simplefilter('ignore', BiopythonParserWarning)

Entrez.email = "tjlngpl2503@gmail.com"

INPUT_FILE = "uniprot_cds.fasta"
OUTPUT_FILE = "uniprot_final_cds.fasta"

# =========================
# CLEAN IDS
# =========================
def clean_id(x):
    if not x or x == "N/A":
        return ""
    x = x.strip().split('.')[0]
    x = x.split("|")[0]
    return x

# =========================
# PARSE FASTA
# =========================
records = list(SeqIO.parse(INPUT_FILE, "fasta"))

parsed_data = []

for rec in records:
    parts = rec.description.split("|")
    parts += [""] * (7 - len(parts))

    p0, p1, p2, p3 = parts[0], parts[1], parts[2], parts[3]

    protein_id = ""
    nucleotide_id = ""
    molecule_type = ""

    # TYPE 2 (NP + NM)
    if p1.startswith("NP_") and p2.startswith("NM_"):
        protein_id = p1
        nucleotide_id = p2
        molecule_type = p3

    # TYPE 1 (Genomic DNA)
    elif p3 == "Genomic_DNA":
        nucleotide_id = p1
        protein_id = p2
        molecule_type = p3

    # TYPE 3 (fallback)
    else:
        protein_id = p1
        nucleotide_id = p2
        molecule_type = p3

    parsed_data.append({
        "record": rec,
        "protein_id": clean_id(protein_id),
        "nucleotide_id": clean_id(nucleotide_id),
        "molecule_type": clean_id(molecule_type)
    })

# =========================
# RESOLVE IDS
# =========================
def resolve_ids(pid, nid, mol):
    if pid.startswith("NM_") and nid.startswith("NP_"):
        return nid, pid

    if pid:
        return pid, nid

    if not pid and mol:
        return mol, nid

    return "", nid

# =========================
# BUILD ID SETS + MAPPING
# =========================
nucleotide_ids = set()
protein_ids = set()
nuc_to_prot = {}

for item in parsed_data:
    pid, nid = resolve_ids(
        item["protein_id"],
        item["nucleotide_id"],
        item["molecule_type"]
    )

    item["resolved_protein"] = pid
    item["resolved_nucleotide"] = nid

    if nid:
        nucleotide_ids.add(nid)
    if pid:
        protein_ids.add(pid)
    if nid and pid:
        nuc_to_prot[nid] = pid  # 🔥 important mapping

print(f"Nucleotide IDs: {len(nucleotide_ids):,}")
print(f"Protein IDs: {len(protein_ids):,}")

# =========================
# FETCH FUNCTION (FIXED)
# =========================
def fetch_batch(ids, db, mapping=None):
    results = {}

    for attempt in range(2):
        try:
            handle = Entrez.efetch(
                db=db,
                id=",".join(ids),
                rettype="gb",
                retmode="text"
            )

            for rec in SeqIO.parse(handle, "genbank"):
                rid = rec.id.split('.')[0]

                transl_table = "error"
                codon_start = "error"

                target_protein = None
                if mapping:
                    target_protein = mapping.get(rid, "")

                for f in rec.features:
                    if f.type != "CDS":
                        continue

                    # 🔥 Match correct CDS for genomic/WGS
                    if target_protein:
                        cds_proteins = f.qualifiers.get("protein_id", [])
                        cds_proteins = [x.split('.')[0] for x in cds_proteins]

                        if target_protein in cds_proteins:
                            transl_table = f.qualifiers.get("transl_table", ["1"])[0]
                            codon_start = f.qualifiers.get("codon_start", ["1"])[0]
                            break

                    else:
                        transl_table = f.qualifiers.get("transl_table", ["1"])[0]
                        codon_start = f.qualifiers.get("codon_start", ["1"])[0]
                        break

                results[rid] = (transl_table, codon_start)

            handle.close()
            return results

        except Exception:
            time.sleep(0.5)

    return results

# =========================
# PARALLEL FETCH
# =========================
def process_ids_parallel(id_list, db, mapping=None):
    final = {}

    if not id_list:
        return final

    BATCH_SIZE = 200
    MAX_WORKERS = 5

    batches = [id_list[i:i+BATCH_SIZE] for i in range(0, len(id_list), BATCH_SIZE)]

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [
            executor.submit(fetch_batch, batch, db, mapping)
            for batch in batches
        ]

        for f in tqdm(as_completed(futures), total=len(futures), desc=f"{db} fetching"):
            final.update(f.result())

    return final

# =========================
# FETCH DATA
# =========================
print("\nFetching nucleotide data...")
nucleotide_map = process_ids_parallel(
    list(nucleotide_ids),
    "nucleotide",
    nuc_to_prot   # for genomic matching
)

print("\nFetching protein data...")
protein_map = process_ids_parallel(
    list(protein_ids),
    "protein"
)

# =========================
# WRITE FASTA OUTPUT
# =========================
print("\nWriting FASTA output...")

with open(OUTPUT_FILE, "w") as out:

    for item in tqdm(parsed_data):
        rec = item["record"]
        nid = item["resolved_nucleotide"]
        pid = item["resolved_protein"]

        transl_table = "error"
        codon_start = "error"

        if nid and nid in nucleotide_map:
            transl_table, codon_start = nucleotide_map[nid]

        elif pid and pid in protein_map:
            transl_table, codon_start = protein_map[pid]

        new_header = (
            rec.description
            + f"|transl_table={transl_table}"
            + f"|codon_start={codon_start}"
        )

        out.write(f">{new_header}\n")
        out.write(str(rec.seq) + "\n")

print(f"\n Done → {OUTPUT_FILE}")

In [ ]:
# substring matching of peptides to cds

import pandas as pd
import itertools
import ahocorasick
from tqdm import tqdm
from collections import defaultdict
from Bio.Data import CodonTable

# ==============================
# CONFIG
# ==============================
DNA_FASTA = "uniprot_final_cds.fasta"
PEPTIDE_FILE = "absent_pep.csv"
OUTPUT_FILE = "pep_hit.csv"

DNA_LENGTH = 15

# ==============================
# HEADER PARSER (FIXED HARD)
# ==============================
def parse_header(header):
    parts = header.strip().split("|")

    info = {
        "uniprot_id": parts[0] if len(parts) > 0 else "",
        "nucleotide_id": parts[1] if len(parts) > 1 else "",
        "protein_id": parts[2] if len(parts) > 2 else "",
        "molecule_type": parts[3] if len(parts) > 3 else "",
        "status": parts[4] if len(parts) > 4 else "",
        "uniprot_length": parts[5] if len(parts) > 5 else "",
        "ncbi_length": parts[6] if len(parts) > 6 else "",
        "transl_table": "1",
        "codon_start": "1"
    }

    for p in parts:
        if p.startswith("transl_table="):
            val = p.split("=")[1]

            # FIX: ensure numeric
            if val.isdigit():
                info["transl_table"] = val
            else:
                info["transl_table"] = "1"

        elif p.startswith("codon_start="):
            val = p.split("=")[1]

            # optional safety
            if val.isdigit():
                info["codon_start"] = val
            else:
                info["codon_start"] = "1"

    return info

# ==============================
# FASTA READER
# ==============================
def read_fasta(file):
    with open(file, "r") as f:
        header, seq = "", ""
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                if seq:
                    yield header, seq
                header = line[1:]
                seq = ""
            else:
                seq += line.upper()
        if seq:
            yield header, seq

# ==============================
# GET CODON TABLE
# ==============================
def get_codon_map(table_id):
    try:
        table = CodonTable.unambiguous_dna_by_id[int(table_id)]
    except:
        table = CodonTable.unambiguous_dna_by_id[1]

    codon_map = defaultdict(list)

    for codon, aa in table.forward_table.items():
        codon_map[aa].append(codon)

    return codon_map

# ==============================
# PEPTIDE → DNA VARIANTS
# ==============================
def peptide_to_dna(peptide, codon_map):
    codon_lists = [codon_map[aa] for aa in peptide if aa in codon_map]
    for combo in itertools.product(*codon_lists):
        yield "".join(combo)

# ==============================
# BUILD AUTOMATON
# ==============================
def build_automaton(peptides, codon_map):
    A = ahocorasick.Automaton()

    for peptide in tqdm(peptides, desc="Generating 15-mers"):
        for dna_seq in peptide_to_dna(peptide, codon_map):
            A.add_word(dna_seq, (peptide, dna_seq))

    A.make_automaton()
    return A

# ==============================
# SAFE SORT FUNCTION (NEW)
# ==============================
def safe_int(x):
    try:
        return int(x)
    except:
        return 999  # push bad values to end

# ==============================
# MAIN
# ==============================
def main():
    print("Reading peptides...")
    peptides_df = pd.read_csv(PEPTIDE_FILE)
    peptide_list = peptides_df.iloc[:, 0].astype(str).str.upper().str.strip().tolist()

    print("Grouping FASTA by transl_table...")
    fasta_groups = defaultdict(list)

    for header, seq in read_fasta(DNA_FASTA):
        info = parse_header(header)
        fasta_groups[info["transl_table"]].append((header, seq, info))

    results = []

    # =========================================
    # ORDERED PROCESSING (SAFE)
    # =========================================
    for tt in sorted(fasta_groups.keys(), key=safe_int):

        entries = fasta_groups[tt]

        print(f"\n==============================")
        print(f"Processing transl_table = {tt}")
        print(f"Sequences = {len(entries)}")
        print(f"==============================")

        codon_map = get_codon_map(tt)

        print("Building automaton...")
        automaton = build_automaton(peptide_list, codon_map)

        print("Scanning sequences...")

        for header, seq, header_info in tqdm(entries, desc=f"Table {tt}"):

            hit_dict = {}

            for end_idx, (peptide, dna_seq) in automaton.iter(seq):
                start_idx = end_idx - DNA_LENGTH + 1

                if peptide not in hit_dict:
                    hit_dict[peptide] = {
                        "positions": [],
                        "dna_matches": set()
                    }

                hit_dict[peptide]["positions"].append(start_idx)
                hit_dict[peptide]["dna_matches"].add(dna_seq)

            for peptide, info in hit_dict.items():
                results.append({
                    **header_info,
                    "peptide": peptide,
                    "matched_15mer": ";".join(info["dna_matches"]),
                    "cds_sequence": seq,
                    "num_hits": len(info["positions"]),
                    "positions": ";".join(map(str, info["positions"]))
                })

    print("\nSaving output...")
    pd.DataFrame(results).to_csv(OUTPUT_FILE, index=False)

    print("Done!")

# ==============================
# RUN
# ==============================
if __name__ == "__main__":
    main()

In [ ]:
# orf analysis

import pandas as pd
from tqdm import tqdm

# =========================
# LOAD
# =========================
df = pd.read_csv("pep_hit.csv", low_memory=False)

# =========================
# CLEAN
# =========================
df["codon_start"] = pd.to_numeric(df.get("codon_start", 1), errors="coerce").fillna(1).astype(int)

# 0-based offset
df["codon_offset"] = df["codon_start"] - 1

# Ensure position exists
df["positions"] = pd.to_numeric(df["positions"], errors="coerce")

# =========================
# INIT OUTPUT
# =========================
df["frame_status"] = ""
df["shift"] = pd.NA

# =========================
# HELPER: decide if codon_start should be used
# =========================
def use_codon_start(pid):
    if pd.isna(pid) or str(pid).strip() == "":
        return True

    pid = str(pid)

    if pid.startswith("NM_") or pid.startswith("NP_"):
        return True

    return False

# =========================
# FRAME CHECK
# =========================
for row in tqdm(df.itertuples(), total=len(df), desc="Frame check"):

    i = row.Index

    if pd.isna(row.positions):
        continue

    pos0 = int(row.positions)

    if use_codon_start(row.protein_id):
        shift = (pos0 - int(row.codon_offset)) % 3
    else:
        shift = pos0 % 3

    if shift == 0:
        df.at[i, "frame_status"] = "in_frame"
        df.at[i, "shift"] = 0
    else:
        df.at[i, "frame_status"] = "frameshift"
        df.at[i, "shift"] = shift

# =========================
# SAVE
# =========================
df.to_csv("pep_frames.csv", index=False)

print("Frame-only analysis complete")

In [ ]:
# validating inframe hits
import pandas as pd
from tqdm import tqdm
import requests
import time
import csv
from Bio import Entrez, SeqIO

# =========================
# CONFIG
# =========================
INPUT_CSV = "pep_frames.csv"

OUTPUT_ALL = "inframe_full_validation.csv"
OUTPUT_YES = "inframe_yes_only.csv"

Entrez.email = "tejalnagpal2504@gmail.com"   # REQUIRED


# =========================
# CLEAN ID
# =========================
def clean_id(x):

    if pd.isna(x):
        return None

    x = str(x).strip()

    if x == "":
        return None

    return x


# =========================
# GROUP-BASED ID LOGIC
# =========================
def resolve_protein_id(row):

    pid = clean_id(row.get("protein_id"))
    nid = clean_id(row.get("nucleotide_id"))
    mol = clean_id(row.get("molecule_type"))

    # grp1 → proper protein_id
    if pid and not pid.startswith("NM_"):
        return pid

    # grp2 → NM in protein_id, NP in nucleotide_id
    if pid and pid.startswith("NM_") and nid and nid.startswith("NP_"):
        return nid

    # grp3 → blank protein_id → use molecule_type
    if (not pid) and mol:
        return mol

    return None


# =========================
# FETCH PROTEINS (NCBI)
# =========================
def fetch_protein_batch(ids):

    results = {}

    for pid in ids:

        url = (
            "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
            f"efetch.fcgi?db=protein&id={pid}&rettype=fasta&retmode=text"
        )

        try:

            r = requests.get(url, timeout=30)

            if r.status_code != 200:
                continue

            if not r.text.startswith(">"):
                continue

            seq = "".join(r.text.split("\n")[1:]).strip()

            if seq:
                results[pid] = seq

        except Exception as e:
            print(f"Protein fetch error {pid}: {e}")

        time.sleep(0.2)

    return results


# =========================
# FETCH GENBANK ANNOTATIONS
# =========================
def fetch_genbank_annotations(nuc_id):

    notes = []
    exceptions = []

    try:

        handle = Entrez.efetch(
            db="nucleotide",
            id=nuc_id,
            rettype="gb",
            retmode="text"
        )

        record = SeqIO.read(handle, "genbank")

        handle.close()

        for feature in record.features:

            if feature.type == "CDS":

                if "note" in feature.qualifiers:
                    notes.extend(feature.qualifiers["note"])

                if "exception" in feature.qualifiers:
                    exceptions.extend(feature.qualifiers["exception"])

    except Exception as e:
        print(f"GenBank fetch error {nuc_id}: {e}")

    return "; ".join(notes), "; ".join(exceptions)


# =========================
# LOAD CSV
# =========================
print("Loading dataset...")

df = pd.read_csv(INPUT_CSV, low_memory=False)

print(f"Total rows: {len(df)}")


# =========================
# FILTER IN-FRAME ONLY
# =========================
print("\nFiltering in-frame entries...")

df["shift"] = pd.to_numeric(df["shift"], errors="coerce")

df = df[
    (df["frame_status"].astype(str).str.lower() == "in_frame") &
    (df["shift"] == 0)
].copy()

print(f"After in-frame filter: {len(df)}")


# =========================
# RESOLVE PROTEIN IDS
# =========================
print("\nResolving protein IDs...")

df["protein_id_clean"] = df.apply(resolve_protein_id, axis=1)

df = df[df["protein_id_clean"].notna()].copy()

print(f"Valid protein IDs: {len(df)}")


# =========================
# FETCH PROTEINS
# =========================
print("\nFetching protein sequences...")

unique_proteins = list(set(df["protein_id_clean"]))

protein_db = {}

BATCH = 20

for i in tqdm(
    range(0, len(unique_proteins), BATCH),
    desc="Protein fetch"
):

    batch = unique_proteins[i:i+BATCH]

    protein_db.update(fetch_protein_batch(batch))

print(f"Fetched {len(protein_db)} proteins")


# =========================
# MAP PROTEINS
# =========================
df["protein_sequence"] = df["protein_id_clean"].map(protein_db)


# =========================
# FETCH GENBANK ANNOTATIONS
# =========================
print("\nFetching GenBank annotations...")

unique_nuc = list(set(df["nucleotide_id"].dropna()))

annotation_db = {}

for nid in tqdm(unique_nuc, desc="GenBank fetch"):

    notes, exceptions = fetch_genbank_annotations(nid)

    annotation_db[nid] = {
        "note": notes,
        "exception": exceptions
    }

    time.sleep(0.3)


# =========================
# MAP ANNOTATIONS
# =========================
df["note"] = df["nucleotide_id"].map(
    lambda x: annotation_db.get(x, {}).get("note", "")
)

df["exception"] = df["nucleotide_id"].map(
    lambda x: annotation_db.get(x, {}).get("exception", "")
)


# =========================
# VALIDATE PEPTIDES
# =========================
print("\nValidating peptides...")

new_cols = []

for row in tqdm(
    df.itertuples(),
    total=len(df),
    desc="Processing"
):

    peptide = getattr(row, "peptide", "")
    protein_seq = getattr(row, "protein_sequence", "")
    cds_seq = getattr(row, "cds_sequence", "")

    # peptide check
    found = False

    if isinstance(protein_seq, str) and protein_seq:

        found = peptide in protein_seq

    # lengths
    prot_len = len(protein_seq) if isinstance(protein_seq, str) else 0
    cds_len = len(cds_seq) if isinstance(cds_seq, str) else 0

    if cds_len and prot_len:

        length_flag = (
            "OK"
            if abs(prot_len - cds_len/3) <= 1
            else "Mismatch"
        )

    else:
        length_flag = "Unknown"

    new_cols.append({
        "Peptide_in_Protein": "YES" if found else "NO",
        "Protein_Length": prot_len,
        "CDS_Length": cds_len,
        "Length_Check": length_flag
    })


# =========================
# FINAL DATAFRAME
# =========================
df_final = pd.concat(
    [df.reset_index(drop=True), pd.DataFrame(new_cols)],
    axis=1
)


# =========================
# CLEAN TEXT COLUMNS
# =========================
print("\nCleaning text columns...")

text_cols = ["note", "exception"]

for col in text_cols:

    df_final[col] = (
        df_final[col]
        .astype(str)
        .str.replace("\n", " ", regex=False)
        .str.replace("\r", " ", regex=False)
        .str.replace("\t", " ", regex=False)
    )


# =========================
# FILTER YES ONLY
# =========================
df_yes = df_final[
    df_final["Peptide_in_Protein"] == "YES"
].copy()


# =========================
# SAVE FILES
# =========================
print("\nSaving files...")

df_final.to_csv(
    OUTPUT_ALL,
    index=False,
    quoting=csv.QUOTE_ALL
)

df_yes.to_csv(
    OUTPUT_YES,
    index=False,
    quoting=csv.QUOTE_ALL
)


# =========================
# DONE
# =========================
print("\n✅ DONE")

print("\nSaved files:")
print(f" - {OUTPUT_ALL}")
print(f" - {OUTPUT_YES}")

In [ ]:
# perfomed manual checks to the hits not found in ncbi. inframe_full_validation.csv file edited and renamed to inframe_full_validation_tranls.csv

In [ ]:
# making edits in the orf file based on the validation

import pandas as pd

# =========================
# INPUT FILES
# =========================
input1 = "inframe_full_validation_tranls.csv"
input2 = "pep_frames.csv"
input3 = "absent_pep.csv"

# =========================
# OUTPUT FILES
# =========================
output_cleaned = "final_pep_frames_tranls.csv"
output_yes_removed = "peptides_in_ncbi_tranls.csv"
output_absent_cleaned = "final_absent_pep_tranls.csv"
output_stats = "analysis.txt"

# =========================
# LOAD FILES
# =========================
df1 = pd.read_csv(input1, low_memory=False)
df2 = pd.read_csv(input2, low_memory=False)
df3 = pd.read_csv(input3, low_memory=False)

# =========================
# CLEAN
# =========================
df1.columns = df1.columns.str.strip()
df2.columns = df2.columns.str.strip()
df3.columns = df3.columns.str.strip()

df1 = df1.fillna("")
df2 = df2.fillna("")
df3 = df3.fillna("")

# normalize columns
for col in ["uniprot_id", "protein_id", "nucleotide_id", "peptide", "matched_15mer"]:
    if col in df1.columns:
        df1[col] = df1[col].astype(str).str.strip()
    if col in df2.columns:
        df2[col] = df2[col].astype(str).str.strip()
    if col in df3.columns:
        df3[col] = df3[col].astype(str).str.strip()

df2["shift"] = df2["shift"].astype(str)

# =========================
# SPLIT YES / NO
# =========================
yes_df = df1[df1["Peptide_in_Protein"].str.upper() == "YES"].copy()
no_df  = df1[df1["Peptide_in_Protein"].str.upper() == "NO"].copy()

# =========================
# PROCESS YES PEPTIDES (ONLY LIST)
# =========================
yes_peptides = yes_df["peptide"].dropna().unique()

pd.DataFrame(yes_peptides, columns=["peptide"]).to_csv(
    output_yes_removed, index=False
)

# remove YES peptides from df2
df2_filtered = df2[~df2["peptide"].isin(yes_peptides)].copy()

# remove YES peptides from df3 (absent_pep.csv)
df3_filtered = df3[~df3["peptide"].isin(yes_peptides)].copy()
df3_filtered.to_csv(output_absent_cleaned, index=False)

# =========================
# STRICT ROW MATCH FUNCTION
# =========================
def build_mask(df, row):
    return (
        (df["uniprot_id"] == row["uniprot_id"]) &
        (df["protein_id"] == row["protein_id"]) &
        (df["peptide"] == row["peptide"]) &
        (df["matched_15mer"] == row["matched_15mer"])
    )

# =========================
# RULE TERMS
# =========================
seq_error_terms = ["seq error", "sequencing"]
frameshift_terms = ["frameshift", "mutation"]
transl_issue_terms = ["auaa", "v and l"]
download_issue_terms = ["download", "ambiguous"]

# =========================
# COUNTERS
# =========================
removed_no_count = 0
frameshift_modified = 0
transl_issue_modified = 0
download_issue_modified = 0

remove_no_indices = []

# =========================
# APPLY RULES (ROW LEVEL)
# =========================
for _, row in no_df.iterrows():

    note = str(row.get("Self_notes", "")).lower().strip()

    mask = build_mask(df2_filtered, row)
    matched_indices = df2_filtered[mask].index.tolist()

    if not matched_indices:
        print(f"⚠️ No match → {row['uniprot_id']} | {row['peptide']}")
        continue

    # REMOVE SEQ ERRORS
    if any(term in note for term in seq_error_terms):
        remove_no_indices.extend(matched_indices)

    # FRAMESHIFT / MUTATION
    elif any(term in note for term in frameshift_terms):

        frameshift_modified += len(matched_indices)

        df2_filtered.loc[mask, "frame_status"] = "frameshift"
        df2_filtered.loc[mask, "shift"] = "1"

    # TRANSL TABLE ISSUE
    elif any(term in note for term in transl_issue_terms):

        transl_issue_modified += len(matched_indices)

        df2_filtered.loc[mask, "shift"] = "transl_table_issue"

    # DOWNLOAD ISSUE
    elif any(term in note for term in download_issue_terms):

        download_issue_modified += len(matched_indices)

        df2_filtered.loc[mask, "frame_status"] = "frameshift"
        df2_filtered.loc[mask, "shift"] = "1"

# =========================
# REMOVE NO SEQ ERRORS
# =========================
remove_no_indices = list(set(remove_no_indices))
removed_no_count = len(remove_no_indices)

df2_filtered = df2_filtered.drop(index=remove_no_indices).copy()

# =========================
# SAVE OUTPUT
# =========================
df2_filtered.to_csv(output_cleaned, index=False)

# =========================
# SUMMARY
# =========================
summary = f"""
=========================
FINAL ANALYSIS SUMMARY
=========================

YES peptides saved: {len(yes_peptides)}
NO seq-error removed: {removed_no_count}

Frameshift modified: {frameshift_modified}
Translation issue modified: {transl_issue_modified}
Download issue modified: {download_issue_modified}

-----------------------------------

Remaining entries: {len(df2_filtered)}
Unique peptides: {df2_filtered['peptide'].nunique()}
Unique UniProt: {df2_filtered['uniprot_id'].nunique()}
Unique 15mers: {df2_filtered['matched_15mer'].nunique()}

-----------------------------------

Absent-peptide file ({input3}):
Original rows: {len(df3)}
Rows after removing YES peptides: {len(df3_filtered)}
Rows removed: {len(df3) - len(df3_filtered)}
Unique peptides remaining: {df3_filtered['peptide'].nunique()}

=========================
"""

print(summary)

with open(output_stats, "w") as f:
    f.write(summary)

print("✅ Processing complete.")

In [ ]:
# adding taxonomy information

import pandas as pd
import requests
from tqdm import tqdm
import time

# ================= SETTINGS =================
input_file = "final_pep_frames_tranls.csv"
output_file = "pep_taxonomy.csv"

BATCH_SIZE = 100
BASE_URL = "https://rest.uniprot.org/uniprotkb/search"

# ================= LOAD =================
df = pd.read_csv(input_file, dtype=str).fillna("")

# ================= UNIQUE IDS =================
unique_ids = list(set(df["uniprot_id"].str.strip()) - {""})
print(f"Unique UniProt IDs: {len(unique_ids):,}")

# ================= FETCH FUNCTION =================
def fetch_batch(batch):

    query = " OR ".join([f"accession:{x}" for x in batch])

    params = {
        "query": query,
        "format": "json",
        "fields": "accession,organism_name,lineage",
        "size": 500
    }

    try:
        r = requests.get(BASE_URL, params=params, timeout=20)

        if r.status_code != 200:
            print(f"HTTP error: {r.status_code}")
            return {}

        data = r.json()
        result_map = {}

        for entry in data.get("results", []):

            uid = entry.get("primaryAccession")

            organism = entry.get("organism", {})
            species = organism.get("scientificName", None)
            lineage = organism.get("lineage", [])

            domain = lineage[0] if len(lineage) > 0 else None
            phylum = lineage[1] if len(lineage) > 1 else None
            class_rank = lineage[2] if len(lineage) > 2 else None
            order = lineage[3] if len(lineage) > 3 else None

            result_map[uid] = (species, domain, phylum, class_rank, order)

        return result_map

    except Exception as e:
        print("Exception:", e)
        return {}

# ================= MAIN LOOP =================
results_map = {}

for i in tqdm(range(0, len(unique_ids), BATCH_SIZE), desc="Batch fetching"):
    batch = unique_ids[i:i+BATCH_SIZE]

    res = fetch_batch(batch)
    results_map.update(res)

    # ================= LIVE PREVIEW =================
    if res:
        print("\n Batch sample:")
        for k in list(res.keys())[:3]:
            print(f"{k} → {res[k]}")
    else:
        print("\n Empty batch returned")

    time.sleep(0.1)

# ================= MAP BACK =================
def assign(uid):
    return results_map.get(uid, (None, None, None, None, None))

df[["Species", "Domain", "Phylum", "Class", "Order"]] = df["uniprot_id"].apply(
    lambda x: pd.Series(assign(x))
)

# ================= DEBUG SUMMARY =================
mapped = df["Species"].notna().sum()
total = len(df)

print(f"\n✅ Mapping success: {mapped}/{total} ({(mapped/total)*100:.2f}%)")

# ================= PREVIEW =================
print("\n🔍 First 10 rows:")
print(df[["uniprot_id", "Species", "Domain", "Phylum", "Class", "Order"]].head(10))

# ================= SAVE =================
df.to_csv(output_file, index=False)

print("\n✅ Done!")
print("Saved as:", output_file)

In [ ]:
# adding GO information

import pandas as pd
import requests
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import os
import time

# =========================================
# CONFIG
# =========================================
INPUT_FILE = "pep_taxonomy.csv"

OUTPUT_FILE = "pep_GO.csv"

CHECKPOINT_FILE = "go_checkpoint.csv"

BASE_URL = "https://rest.uniprot.org/uniprotkb/"

MAX_WORKERS = 4
MAX_RETRIES = 5

# =========================================
# CHUNK SETTINGS
# =========================================
CHUNKSIZE = 100000

# =========================================
# REQUEST SESSION
# =========================================
session = requests.Session()

# =========================================
# LOAD OLD CHECKPOINT
# =========================================
results_dict = {}

if os.path.exists(CHECKPOINT_FILE):

    print("\nLoading previous checkpoint...")

    old_df = pd.read_csv(
        CHECKPOINT_FILE,
        low_memory=False
    )

    for row in old_df.itertuples():

        results_dict[row.uniprot_id] = (
            row.GO_Biological_Process,
            row.GO_Molecular_Function,
            row.GO_Cellular_Component
        )

    print(
        f"Recovered {len(results_dict)} completed IDs"
    )

# =========================================
# FETCH FUNCTION
# =========================================
def fetch_go(uid):

    url = BASE_URL + uid + ".json"

    for attempt in range(MAX_RETRIES):

        try:

            r = session.get(
                url,
                timeout=20
            )

            if r.status_code != 200:

                time.sleep(1)
                continue

            data = r.json()

            process_terms = []
            function_terms = []
            component_terms = []

            # =================================
            # EXTRACT GO TERMS
            # =================================
            for xref in data.get(
                "uniProtKBCrossReferences",
                []
            ):

                if xref.get("database") == "GO":

                    for p in xref.get(
                        "properties",
                        []
                    ):

                        if p.get("key") == "GoTerm":

                            term = p.get("value")

                            if term.startswith("P:"):

                                process_terms.append(
                                    term[2:]
                                )

                            elif term.startswith("F:"):

                                function_terms.append(
                                    term[2:]
                                )

                            elif term.startswith("C:"):

                                component_terms.append(
                                    term[2:]
                                )

            return uid, (
                "; ".join(process_terms),
                "; ".join(function_terms),
                "; ".join(component_terms),
                True
            )

        except Exception as e:

            print(
                f"Retry {attempt+1} "
                f"for {uid}: {e}"
            )

            time.sleep(
                2 * (attempt + 1)
            )

    # =====================================
    # FAILED AFTER RETRIES
    # =====================================
    return uid, (
        None,
        None,
        None,
        False
    )

# =========================================
# COUNT TOTAL ROWS
# =========================================
print("\nCounting total rows...")

total_rows = (
    sum(1 for _ in open(INPUT_FILE)) - 1
)

total_chunks = (
    total_rows // CHUNKSIZE
) + 1

print(f"Total rows: {total_rows:,}")
print(f"Total chunks: {total_chunks:,}")

# =========================================
# OUTPUT HEADER FLAG
# =========================================
first_write = True

# =========================================
# SUMMARY COUNTS
# =========================================
total_success = 0
total_fail = 0

# =========================================
# PROCESS INPUT FILE IN CHUNKS
# =========================================
for chunk in tqdm(
    pd.read_csv(
        INPUT_FILE,
        chunksize=CHUNKSIZE,
        low_memory=False
    ),
    total=total_chunks,
    desc="Processing Chunks"
):

    # =====================================
    # CLEAN IDS
    # =====================================
    chunk["uniprot_id"] = (
        chunk["uniprot_id"]
        .astype(str)
        .str.strip()
    )

    # =====================================
    # UNIQUE IDS IN CHUNK
    # =====================================
    unique_ids = (
        chunk["uniprot_id"]
        .dropna()
        .unique()
        .tolist()
    )

    # =====================================
    # FIND REMAINING IDS
    # =====================================
    remaining_ids = [
        uid
        for uid in unique_ids
        if uid not in results_dict
    ]

    print(
        f"\nChunk rows: {len(chunk):,}"
    )

    print(
        f"Unique IDs in chunk: "
        f"{len(unique_ids):,}"
    )

    print(
        f"New IDs to fetch: "
        f"{len(remaining_ids):,}"
    )

    # =====================================
    # PARALLEL FETCH
    # =====================================
    new_rows = []

    with ThreadPoolExecutor(
        max_workers=MAX_WORKERS
    ) as executor:

        futures = [
            executor.submit(
                fetch_go,
                uid
            )
            for uid in remaining_ids
        ]

        for f in tqdm(
            as_completed(futures),
            total=len(futures),
            desc="Fetching GO",
            leave=False
        ):

            uid, result = f.result()

            bp, mf, cc, success = result

            if success:

                results_dict[uid] = (
                    bp,
                    mf,
                    cc
                )

                total_success += 1

                checkpoint_row = {
                    "uniprot_id": uid,
                    "GO_Biological_Process": bp,
                    "GO_Molecular_Function": mf,
                    "GO_Cellular_Component": cc
                }

                new_rows.append(
                    checkpoint_row
                )

            else:

                total_fail += 1

    # =====================================
    # SAVE CHECKPOINT
    # =====================================
    if len(new_rows) > 0:

        checkpoint_df = pd.DataFrame(
            new_rows
        )

        if os.path.exists(
            CHECKPOINT_FILE
        ):

            checkpoint_df.to_csv(
                CHECKPOINT_FILE,
                mode="a",
                header=False,
                index=False
            )

        else:

            checkpoint_df.to_csv(
                CHECKPOINT_FILE,
                index=False
            )

    # =====================================
    # MAP GO TERMS TO CURRENT CHUNK
    # =====================================
    chunk["GO_Biological_Process"] = (
        chunk["uniprot_id"]
        .map(
            lambda x:
            results_dict.get(
                x,
                (None, None, None)
            )[0]
        )
    )

    chunk["GO_Molecular_Function"] = (
        chunk["uniprot_id"]
        .map(
            lambda x:
            results_dict.get(
                x,
                (None, None, None)
            )[1]
        )
    )

    chunk["GO_Cellular_Component"] = (
        chunk["uniprot_id"]
        .map(
            lambda x:
            results_dict.get(
                x,
                (None, None, None)
            )[2]
        )
    )

    # =====================================
    # SAVE CHUNK OUTPUT
    # =====================================
    if first_write:

        chunk.to_csv(
            OUTPUT_FILE,
            index=False,
            mode="w"
        )

        first_write = False

    else:

        chunk.to_csv(
            OUTPUT_FILE,
            index=False,
            mode="a",
            header=False
        )

    print(
        f"Chunk saved to "
        f"{OUTPUT_FILE}"
    )

# =========================================
# FINAL SUMMARY
# =========================================
print("\n==============================")
print("GO ANNOTATION COMPLETE")
print("==============================")
print(
    f"\nSuccessful IDs: "
    f"{total_success}"
)
print(
    f"Failed IDs: "
    f"{total_fail}"
)
print(
    f"\nCheckpoint saved:"
)
print(CHECKPOINT_FILE)
print(
    f"\nFinal output:"
)
print(OUTPUT_FILE)

In [ ]:
# ============================================================
# GOOGLE COLAB CODE
# Ambush Hypothesis Analysis for 15-mers
#
# This code:
# 1. Reads CDS sequences from CSV
# 2. Uses Biopython translation tables
# 3. Checks stop codons in:
#       original frame
#       +1 shifted frame
#       +2 shifted frame
# 4. Counts mapped 15mers for +1 and +2 shifts
# 5. Finds stop codons before the first mapped 15mer
# 6. Finds stop codons inside mapped 15mers
# 7. Performs paired t-test + Bonferroni correction
# 8. Saves detailed results, summary files, and inferential statistics
# ============================================================

# ============================================================
# INSTALL DEPENDENCIES
# ============================================================

!pip install biopython statsmodels -q

# ============================================================
# IMPORTS
# ============================================================

import pandas as pd
import numpy as np
from tqdm import tqdm
from collections import defaultdict
from Bio.Data import CodonTable
import re

from scipy.stats import ttest_rel
from statsmodels.stats.multitest import multipletests
from itertools import combinations

# ============================================================
# FILE PATHS
# ============================================================

input_file = "pep_frames.csv"

output_file = "ambush_hypothesis_results_with_first15mer.csv"
summary_output_file = "ambush_hypothesis_summary_with_first15mer.csv"

mapped15mer_output_file = "mapped15mer_stopcodon_analysis.csv"
mapped15mer_summary_file = "mapped15mer_stopcodon_summary.csv"

inferential_output_file = "stop_count_ttest_bonferroni_results_15mer.csv"

# ============================================================
# CLEAN FUNCTIONS
# ============================================================

def clean_sequence(seq):
    seq = str(seq).upper()
    seq = seq.replace("U", "T")
    seq = re.sub(r"[^ATGC]", "", seq)
    return seq


def normalize_shift(x):
    x = str(x).strip()

    if x in ["1", "1.0"]:
        return "1"

    if x in ["2", "2.0"]:
        return "2"

    if x in ["transl_table_issue", "trans_table_issue"]:
        return "transl_table_issue"

    return x


def normalize_transl_table(x):
    try:
        return int(float(str(x).strip()))
    except Exception:
        return None


def clean_position_plus_one(pos):
    try:
        pos = str(pos).strip()

        if "|" in pos:
            pos = pos.split("|")[0]

        return int(float(pos)) + 1

    except Exception:
        return None


def get_shifted_codons(seq, shift):
    shifted_seq = seq[shift:]
    codons = []

    for i in range(0, len(shifted_seq) - 2, 3):
        codon = shifted_seq[i:i + 3]

        if len(codon) == 3:
            codons.append(codon)

    return codons


def find_stop_codons(codons, stop_codons):
    stop_positions = []
    stop_codon_list = []

    for idx, codon in enumerate(codons):
        if codon in stop_codons:
            stop_positions.append(idx + 1)
            stop_codon_list.append(codon)

    return stop_codon_list, stop_positions


def count_stop_codons_before_nt_position(seq, frame_shift, stop_codons, nt_position):
    """
    Counts stop codons before the first mapped 15mer.

    nt_position is 1-based nucleotide position.
    frame_shift:
        0 = original frame
        1 = +1 frame
        2 = +2 frame
    """

    if nt_position is None or pd.isna(nt_position):
        return 0, "", ""

    codons = get_shifted_codons(seq, frame_shift)

    stop_count = 0
    stop_codon_list = []
    stop_position_list = []

    for idx, codon in enumerate(codons):
        codon_start_0based = frame_shift + idx * 3
        codon_start_1based = codon_start_0based + 1

        if codon_start_1based >= nt_position:
            break

        if codon in stop_codons:
            stop_count += 1
            stop_codon_list.append(codon)
            stop_position_list.append(idx + 1)

    return (
        stop_count,
        "|".join(stop_codon_list),
        "|".join(map(str, stop_position_list))
    )


def count_stop_codons_inside_15mer_all_frames(seq15, stop_codons):
    """
    Counts stop codons inside a 15mer across internal frames 0, 1, and 2.
    """

    seq15 = clean_sequence(seq15)

    stop_count = 0
    found_stop_codons = []
    found_internal_frames = []
    found_positions = []

    for frame in [0, 1, 2]:
        codons = get_shifted_codons(seq15, frame)

        for idx, codon in enumerate(codons):
            if codon in stop_codons:
                stop_count += 1
                found_stop_codons.append(codon)
                found_internal_frames.append(str(frame))
                found_positions.append(str(idx + 1))

    return (
        stop_count,
        "|".join(found_stop_codons),
        "|".join(found_internal_frames),
        "|".join(found_positions)
    )


# ============================================================
# LOAD DATA
# ============================================================

print("\nLoading dataset...")

df = pd.read_csv(
    input_file,
    dtype=str,
    low_memory=False
)

print("\nDataset loaded.")
print("Total rows:", len(df))

# ============================================================
# CLEAN DATA
# ============================================================

print("\nCleaning dataset...")

df["shift"] = df["shift"].apply(normalize_shift)
df["cds_sequence_clean"] = df["cds_sequence"].apply(clean_sequence)

# IMPORTANT: this assumes your pentapeptide file has a column named matched_15mer
df["matched_15mer_clean"] = df["matched_15mer"].apply(clean_sequence)

df["transl_table_clean"] = df["transl_table"].apply(normalize_transl_table)
df["position_corrected_plus_one"] = df["positions"].apply(clean_position_plus_one)

print("\nShift distribution:")
print(df["shift"].value_counts(dropna=False))

print("\nTranslation table distribution:")
print(df["transl_table_clean"].value_counts(dropna=False))

# ============================================================
# BUILD UNIQUE CDS TABLE
# ============================================================

print("\nBuilding unique CDS table...")

unique_cds = df[
    ["cds_sequence_clean", "transl_table_clean"]
].drop_duplicates()

unique_cds = unique_cds.dropna(
    subset=["transl_table_clean"]
)

unique_cds["transl_table_clean"] = unique_cds[
    "transl_table_clean"
].astype(int)

print(
    "Unique CDS + translation table combinations:",
    len(unique_cds)
)

# ============================================================
# COUNT MAPPED 15MERS AND FIRST MAPPED 15MER POSITION
# ============================================================

print("\nCounting mapped 15mers and first mapped 15mer position...")

shift1_counts = defaultdict(int)
shift2_counts = defaultdict(int)
transl_table_issue_counts = defaultdict(int)
all_mapped_counts = defaultdict(int)

first_mapped_position = {}

for _, row in tqdm(
    df.iterrows(),
    total=len(df),
    desc="Counting mapped 15mers"
):

    cds = row["cds_sequence_clean"]
    shift = row["shift"]
    pos = row["position_corrected_plus_one"]

    if shift == "1":
        shift1_counts[cds] += 1
        all_mapped_counts[cds] += 1

    elif shift == "2":
        shift2_counts[cds] += 1
        all_mapped_counts[cds] += 1

    elif shift == "transl_table_issue":
        transl_table_issue_counts[cds] += 1
        all_mapped_counts[cds] += 1

    if pos is not None and not pd.isna(pos):
        if cds not in first_mapped_position:
            first_mapped_position[cds] = pos
        else:
            first_mapped_position[cds] = min(
                first_mapped_position[cds],
                pos
            )

# ============================================================
# MAIN CDS-LEVEL AMBUSH ANALYSIS
# ============================================================

print(
    "\nRunning original-frame, +1-frame, +2-frame, "
    "and before-first-15mer stop codon analysis..."
)

results = []
invalid_translation_tables = []

for _, row in tqdm(
    unique_cds.iterrows(),
    total=len(unique_cds),
    desc="CDS-level stop codon analysis"
):

    cds_seq = row["cds_sequence_clean"]
    transl_table_id = row["transl_table_clean"]

    try:
        codon_table = CodonTable.unambiguous_dna_by_id[
            transl_table_id
        ]

        stop_codons = set(codon_table.stop_codons)

    except Exception:
        invalid_translation_tables.append(transl_table_id)
        continue

    # Original frame
    original_codons = get_shifted_codons(cds_seq, 0)
    original_stop_codons, original_stop_positions = find_stop_codons(
        original_codons,
        stop_codons
    )

    # +1 frame
    plus1_codons = get_shifted_codons(cds_seq, 1)
    plus1_stop_codons, plus1_stop_positions = find_stop_codons(
        plus1_codons,
        stop_codons
    )

    # +2 frame
    plus2_codons = get_shifted_codons(cds_seq, 2)
    plus2_stop_codons, plus2_stop_positions = find_stop_codons(
        plus2_codons,
        stop_codons
    )

    # Stop codons before first mapped 15mer
    first_pos = first_mapped_position.get(cds_seq, np.nan)

    original_before_count, original_before_codons, original_before_positions = (
        count_stop_codons_before_nt_position(
            cds_seq,
            0,
            stop_codons,
            first_pos
        )
    )

    plus1_before_count, plus1_before_codons, plus1_before_positions = (
        count_stop_codons_before_nt_position(
            cds_seq,
            1,
            stop_codons,
            first_pos
        )
    )

    plus2_before_count, plus2_before_codons, plus2_before_positions = (
        count_stop_codons_before_nt_position(
            cds_seq,
            2,
            stop_codons,
            first_pos
        )
    )

    results.append({

        "cds_sequence": cds_seq,
        "translation_table": transl_table_id,
        "stop_codons_for_table": "|".join(sorted(stop_codons)),
        "cds_length_nt": len(cds_seq),
        "first_mapped_15mer_nt_position": first_pos,

        # Original frame
        "original_total_codons": len(original_codons),
        "original_stop_count": len(original_stop_codons),
        "original_stop_density_per_100_codons": (
            len(original_stop_codons) / len(original_codons) * 100
        ) if len(original_codons) > 0 else np.nan,
        "original_stop_codons": "|".join(original_stop_codons),
        "original_stop_positions": "|".join(map(str, original_stop_positions)),

        # +1 frame
        "plus1_total_codons": len(plus1_codons),
        "plus1_stop_count": len(plus1_stop_codons),
        "plus1_stop_density_per_100_codons": (
            len(plus1_stop_codons) / len(plus1_codons) * 100
        ) if len(plus1_codons) > 0 else np.nan,
        "plus1_stop_codons": "|".join(plus1_stop_codons),
        "plus1_stop_positions": "|".join(map(str, plus1_stop_positions)),
        "plus1_mapped_15mers": shift1_counts.get(cds_seq, 0),

        # +2 frame
        "plus2_total_codons": len(plus2_codons),
        "plus2_stop_count": len(plus2_stop_codons),
        "plus2_stop_density_per_100_codons": (
            len(plus2_stop_codons) / len(plus2_codons) * 100
        ) if len(plus2_codons) > 0 else np.nan,
        "plus2_stop_codons": "|".join(plus2_stop_codons),
        "plus2_stop_positions": "|".join(map(str, plus2_stop_positions)),
        "plus2_mapped_15mers": shift2_counts.get(cds_seq, 0),

        # Mapped 15mer summary
        "transl_table_issue_mapped_15mers": transl_table_issue_counts.get(cds_seq, 0),

        "total_mapped_15mers_shift1_shift2": (
            shift1_counts.get(cds_seq, 0)
            +
            shift2_counts.get(cds_seq, 0)
        ),

        "total_mapped_15mers_all_tracked_groups": all_mapped_counts.get(cds_seq, 0),

        # Stop codons before first mapped 15mer
        "original_stop_count_before_first_mapped_15mer": original_before_count,
        "original_stop_codons_before_first_mapped_15mer": original_before_codons,
        "original_stop_positions_before_first_mapped_15mer": original_before_positions,

        "plus1_stop_count_before_first_mapped_15mer": plus1_before_count,
        "plus1_stop_codons_before_first_mapped_15mer": plus1_before_codons,
        "plus1_stop_positions_before_first_mapped_15mer": plus1_before_positions,

        "plus2_stop_count_before_first_mapped_15mer": plus2_before_count,
        "plus2_stop_codons_before_first_mapped_15mer": plus2_before_codons,
        "plus2_stop_positions_before_first_mapped_15mer": plus2_before_positions
    })

# ============================================================
# MAPPED 15MER-LEVEL STOP CODON ANALYSIS
# ============================================================

print("\nRunning mapped 15mer-level stop codon analysis...")

mapped15_results = []

for _, row in tqdm(
    df.iterrows(),
    total=len(df),
    desc="Mapped 15mer stop codon analysis"
):

    transl_table_id = row["transl_table_clean"]

    if pd.isna(transl_table_id):
        continue

    try:
        transl_table_id = int(transl_table_id)

        codon_table = CodonTable.unambiguous_dna_by_id[
            transl_table_id
        ]

        stop_codons = set(codon_table.stop_codons)

    except Exception:
        continue

    seq15 = row["matched_15mer_clean"]

    stop_count, stop_codons_found, internal_frames, internal_positions = (
        count_stop_codons_inside_15mer_all_frames(
            seq15,
            stop_codons
        )
    )

    mapped15_results.append({

        "cds_sequence_clean": row["cds_sequence_clean"],
        "matched_15mer_clean": seq15,
        "shift": row["shift"],
        "translation_table": transl_table_id,

        "mapped_15mer_has_stop_codon": stop_count > 0,
        "mapped_15mer_stop_codon_count": stop_count,
        "mapped_15mer_stop_codons": stop_codons_found,
        "mapped_15mer_stop_internal_frames": internal_frames,
        "mapped_15mer_stop_internal_positions": internal_positions
    })

# ============================================================
# SAVE DETAILED OUTPUTS
# ============================================================

results_df = pd.DataFrame(results)
mapped15_df = pd.DataFrame(mapped15_results)

# ============================================================
# INFERENTIAL STATISTICS
# Paired t-test + Bonferroni correction
# Comparing original_stop_count, plus1_stop_count, plus2_stop_count
# ============================================================

stop_count_cols = [
    "original_stop_count",
    "plus1_stop_count",
    "plus2_stop_count"
]

stats_df = results_df[stop_count_cols].copy()

for col in stop_count_cols:
    stats_df[col] = pd.to_numeric(stats_df[col], errors="coerce")

stats_df = stats_df.dropna()

ttest_results = []

for col1, col2 in combinations(stop_count_cols, 2):

    t_stat, p_value = ttest_rel(
        stats_df[col1],
        stats_df[col2],
        nan_policy="omit"
    )

    ttest_results.append({
        "Comparison": f"{col1} vs {col2}",
        "Variable_1": col1,
        "Variable_2": col2,
        "Mean_1": stats_df[col1].mean(),
        "SD_1": stats_df[col1].std(ddof=1),
        "Mean_2": stats_df[col2].mean(),
        "SD_2": stats_df[col2].std(ddof=1),
        "N": len(stats_df),
        "t_statistic": t_stat,
        "raw_p_value": p_value
    })

inferential_df = pd.DataFrame(ttest_results)

reject, corrected_pvals, _, _ = multipletests(
    inferential_df["raw_p_value"],
    alpha=0.05,
    method="bonferroni"
)

inferential_df["bonferroni_corrected_p"] = corrected_pvals
inferential_df["significant_after_bonferroni_95CI"] = reject

inferential_df.to_csv(
    inferential_output_file,
    index=False
)

print("\nInferential statistics saved:")
print(inferential_output_file)

print("\nStop-count paired t-test + Bonferroni results:")
print(inferential_df)

# ============================================================
# SAVE MAIN OUTPUT FILES
# ============================================================

results_df.to_csv(
    output_file,
    index=False
)

mapped15_df.to_csv(
    mapped15mer_output_file,
    index=False
)

print("\nCDS-level detailed output saved:")
print(output_file)

print("\nMapped 15mer-level output saved:")
print(mapped15mer_output_file)

print("\nTotal CDS analyzed:")
print(len(results_df))

print("\nTotal mapped 15mers analyzed:")
print(len(mapped15_df))

# ============================================================
# CDS-LEVEL SUMMARY STATISTICS
# ============================================================

summary_cols = [

    "original_stop_count",
    "plus1_stop_count",
    "plus2_stop_count",

    "original_stop_density_per_100_codons",
    "plus1_stop_density_per_100_codons",
    "plus2_stop_density_per_100_codons",

    "original_stop_count_before_first_mapped_15mer",
    "plus1_stop_count_before_first_mapped_15mer",
    "plus2_stop_count_before_first_mapped_15mer"
]

summary_df = results_df[
    summary_cols
].describe()

summary_df.to_csv(
    summary_output_file
)

print("\nCDS-level summary saved:")
print(summary_output_file)

# ============================================================
# MAPPED 15MER STOP SUMMARY
# ============================================================

total_15mers = len(mapped15_df)

stop_containing_15mers = mapped15_df[
    mapped15_df["mapped_15mer_has_stop_codon"] == True
]

percent_15mers_with_stop = (
    len(stop_containing_15mers)
    / total_15mers
    * 100
    if total_15mers > 0
    else np.nan
)

average_stop_codons_all_15mers = mapped15_df[
    "mapped_15mer_stop_codon_count"
].mean()

average_stop_codons_stop_containing_only = stop_containing_15mers[
    "mapped_15mer_stop_codon_count"
].mean()

mapped15_summary = pd.DataFrame({

    "metric": [

        "total_mapped_15mers",
        "mapped_15mers_with_at_least_one_stop",
        "percent_mapped_15mers_with_stop",
        "average_stop_codons_per_15mer_all",
        "average_stop_codons_per_stop_containing_15mer_only"
    ],

    "value": [

        total_15mers,
        len(stop_containing_15mers),
        percent_15mers_with_stop,
        average_stop_codons_all_15mers,
        average_stop_codons_stop_containing_only
    ]
})

mapped15_summary.to_csv(
    mapped15mer_summary_file,
    index=False
)

print("\nMapped 15mer stop summary saved:")
print(mapped15mer_summary_file)

# ============================================================
# INVALID TRANSLATION TABLES
# ============================================================

if len(invalid_translation_tables) > 0:
    print("\nInvalid / unsupported translation table IDs encountered:")
    print(sorted(set(invalid_translation_tables)))

# ============================================================
# PREVIEW
# ============================================================

print("\nPreview of CDS-level detailed results:")
print(results_df.head())

print("\nCDS-level summary statistics:")
print(summary_df)

print("\nMapped 15mer stop summary:")
print(mapped15_summary)

print("\nInferential statistics:")
print(inferential_df)

In [ ]:
# Reverse Analysis
import pandas as pd
from collections import defaultdict, Counter


def analyze_complete_peptides(file_path, out_dir="."):
    print(f"Loading data from {file_path}...")

    try:
        df = pd.read_csv(file_path, usecols=["peptide", "matched_15mer", "transl_table"], low_memory=False)
    except FileNotFoundError:
        print(f"Error: '{file_path}' not found.")
        return
    except ValueError as e:
        print(f"Error with columns: {e}.")
        return

    df = df.dropna(subset=["peptide", "matched_15mer"])

    print("Mapping sequences and translation tables...")

    pep_data = defaultdict(lambda: defaultdict(set))  # pep -> 15mer -> set(tables)
    pep_to_tables = defaultdict(set)                  # pep -> set(tables)

    for pep, mer15, t_table in zip(df["peptide"], df["matched_15mer"], df["transl_table"]):
        clean_pep = str(pep).strip().upper()
        clean_15mer = str(mer15).strip().upper()

        if pd.notna(t_table):
            clean_t_table = str(t_table).strip().replace(".0", "")
        else:
            clean_t_table = "Unknown"

        pep_data[clean_pep][clean_15mer].add(clean_t_table)
        pep_to_tables[clean_pep].add(clean_t_table)

    peptides = set(pep_to_tables.keys())

    # --- Tracking Variables for Standard Pairs ---
    total_pep_pairs = 0
    std_pairs_with_15mer_match = 0
    std_pairs_sharing_table = 0
    std_shared_table_examples = []

    # --- Tracking Variables for Palindromes ---
    total_palindromes = 0

    # Count palindromic peptides that have at least two distinct 15-mers whose table-sets overlap.
    pal_pep_sharing_table = 0
    palindrome_table_distribution = Counter()

    pal_with_same_15mer = 0
    pal_same_15mer_sharing_table = 0

    pal_with_diff_15mer_pairs = 0
    pal_diff_15mer_sharing_table = 0

    pal_diff_shared_examples = []
    pal_same_15mer_examples = []

    # ============================================================
    # collect detailed rows for CSV export
    # ============================================================
    std_pairs_rows = []          # all standard reverse pairs + metrics
    palindrome_rows = []         # each palindromic peptide + whether it shares tables across its 15-mers
    palindrome_tables_rows = []  # table -> count (from distribution)
    pal_same15_rows = []         # palindromes with palindromic 15-mers (examples)
    pal_diff15_rows = []         # palindromes with reversed 15-mer pairs (examples)

    print("Analyzing sequences...")
    for pep in peptides:
        rev_pep = pep[::-1]

        if rev_pep in peptides:

            # ==========================================
            # SCENARIO 1: PALINDROMIC PENTAPEPTIDES
            # ==========================================
            if pep == rev_pep:
                total_palindromes += 1

                # NEW LOGIC: table-sharing across distinct 15-mers for the same palindromic peptide
                mers = list(pep_data[pep].items())  # [(15mer, set(tables)), ...]
                shared_tables_for_palindrome = set()
                palindrome_has_shared_table = False

                for i in range(len(mers)):
                    m1, tset1 = mers[i]
                    for j in range(i + 1, len(mers)):
                        m2, tset2 = mers[j]
                        shared = tset1.intersection(tset2)
                        if shared:
                            palindrome_has_shared_table = True
                            shared_tables_for_palindrome.update(shared)

                if palindrome_has_shared_table:
                    pal_pep_sharing_table += 1
                    for t in shared_tables_for_palindrome:
                        palindrome_table_distribution[t] += 1

                # Record per-palindrome row for CSV
                palindrome_rows.append({
                    "pep": pep,
                    "num_15mers": len(pep_data[pep]),
                    "tables_union": ",".join(sorted(pep_to_tables[pep])),
                    "has_shared_table_across_15mers": palindrome_has_shared_table,
                    "shared_tables_across_15mers": ",".join(sorted(shared_tables_for_palindrome)) if shared_tables_for_palindrome else ""
                })

                # 15-mer level analysis (original logic kept)
                has_same_15mer = False
                has_diff_15mer_pair = False
                diff_15mer_shared_table = False
                same_15mer_shared_table = False

                for m1, tables1 in pep_data[pep].items():
                    rev_m1 = m1[::-1]

                    if rev_m1 in pep_data[pep]:
                        # Case A: 15-mer is identical (palindromic 15-mer)
                        if m1 == rev_m1:
                            has_same_15mer = True

                            # This is logically equivalent to bool(tables1), but kept to match your structure
                            if tables1.intersection(tables1):
                                same_15mer_shared_table = True

                            if len(pal_same_15mer_examples) < 10:
                                pal_same_15mer_examples.append({"pep": pep, "15mer": m1, "tables": tables1})

                            # Also store all such cases for CSV
                            pal_same15_rows.append({
                                "pep": pep,
                                "15mer": m1,
                                "tables": ",".join(sorted(tables1))
                            })

                        # Case B: 15-mers are different but form a reverse pair (string reverse)
                        elif m1 < rev_m1:
                            has_diff_15mer_pair = True
                            shared = tables1.intersection(pep_data[pep][rev_m1])

                            if shared:
                                diff_15mer_shared_table = True
                                if len(pal_diff_shared_examples) < 10:
                                    pal_diff_shared_examples.append(
                                        {"pep": pep, "15mer_pair": f"{m1} <-> {rev_m1}", "tables": shared}
                                    )

                                pal_diff15_rows.append({
                                    "pep": pep,
                                    "15mer_1": m1,
                                    "15mer_2": rev_m1,
                                    "shared_tables": ",".join(sorted(shared))
                                })

                if has_same_15mer:
                    pal_with_same_15mer += 1
                if same_15mer_shared_table:
                    pal_same_15mer_sharing_table += 1

                if has_diff_15mer_pair:
                    pal_with_diff_15mer_pairs += 1
                if diff_15mer_shared_table:
                    pal_diff_15mer_sharing_table += 1

            # ==========================================
            # SCENARIO 2: STANDARD REVERSED PAIRS
            # ==========================================
            elif pep < rev_pep:
                total_pep_pairs += 1

                has_15mer_match = False
                for m1 in pep_data[pep].keys():
                    if m1[::-1] in pep_data[rev_pep]:
                        has_15mer_match = True
                        break

                if has_15mer_match:
                    std_pairs_with_15mer_match += 1

                tables1 = pep_to_tables[pep]
                tables2 = pep_to_tables[rev_pep]
                shared_tables = tables1.intersection(tables2)

                if shared_tables:
                    std_pairs_sharing_table += 1
                    if len(std_shared_table_examples) < 10:
                        std_shared_table_examples.append({"pair": f"{pep} <-> {rev_pep}", "tables": shared_tables})

                # Record all standard reverse pairs for CSV
                std_pairs_rows.append({
                    "pep": pep,
                    "rev_pep": rev_pep,
                    "pep_tables": ",".join(sorted(tables1)),
                    "rev_pep_tables": ",".join(sorted(tables2)),
                    "shared_tables": ",".join(sorted(shared_tables)) if shared_tables else "",
                    "has_shared_table": bool(shared_tables),
                    "has_reversed_15mer_match": has_15mer_match
                })

    # ==========================================
    # PRINTING RESULTS
    # ==========================================
    print("\n" + "=" * 70)
    print("1. STANDARD REVERSED PENTAPEPTIDE PAIRS")
    print("=" * 70)
    print(f"Total standard reversed pentapeptide pairs: {total_pep_pairs}")
    print(f"-> Out of total, pairs with matching reversed 15-mers: {std_pairs_with_15mer_match}")

    std_pct = (std_pairs_sharing_table / total_pep_pairs * 100) if total_pep_pairs > 0 else 0
    print(f"-> Reversed pair pentapeptides sharing the same codon table: {std_pairs_sharing_table} ({std_pct:.2f}%)")

    print("\nSample of pairs with the same codon tables:")
    if std_shared_table_examples:
        for ex in std_shared_table_examples:
            print(f"  {ex['pair']} (Tables: {', '.join(ex['tables'])})")
    else:
        print("  None found.")

    print("\n" + "=" * 70)
    print("2. PALINDROMIC PENTAPEPTIDES")
    print("=" * 70)
    print(f"Total palindromic pentapeptides: {total_palindromes}")

    pal_pep_pct = (pal_pep_sharing_table / total_palindromes * 100) if total_palindromes > 0 else 0
    print(
        "-> Palindromic pentapeptides with ≥2 distinct 15-mers sharing a codon table: "
        f"{pal_pep_sharing_table} ({pal_pep_pct:.2f}%)"
    )

    print(f"-> Out of total, palindromes with EXACT SAME 15-mers (Palindromic 15-mers): {pal_with_same_15mer}")

    pal_15mer_pct = (pal_same_15mer_sharing_table / pal_with_same_15mer * 100) if pal_with_same_15mer > 0 else 0
    print(f"   -> Do these exact same 15-mers share a codon table?: {pal_same_15mer_sharing_table} ({pal_15mer_pct:.2f}%)")

    print(f"-> Out of total, palindromes with DIFFERENT 15-mers forming reverse pairs: {pal_with_diff_15mer_pairs}")

    pal_diff_pct = (pal_diff_15mer_sharing_table / pal_with_diff_15mer_pairs * 100) if pal_with_diff_15mer_pairs > 0 else 0
    print(
        f"   -> Reversed 15-mer pairs (within palindromes) sharing the same codon table: "
        f"{pal_diff_15mer_sharing_table} ({pal_diff_pct:.2f}%)"
    )

    print("\n" + "=" * 70)
    print("3. PALINDROME CODON TABLE DISTRIBUTION")
    print("=" * 70)
    print("Which specific codon tables do these palindromes (that share tables across 15-mers) live in?")
    if palindrome_table_distribution:
        for table, count in palindrome_table_distribution.most_common():
            print(f"  Table {table}: Contains {count} palindromes")
    else:
        print("  No distribution data found.")

    print("\n" + "-" * 70)
    print("Sample of palindromes with EXACT SAME 15-mers:")
    if pal_same_15mer_examples:
        for ex in pal_same_15mer_examples:
            print(f"  Peptide: {ex['pep']} | 15-mer: {ex['15mer']} (Tables: {', '.join(ex['tables'])})")
    else:
        print("  None found.")

    # ==========================================
    # SAVE RESULTS TO CSV
    # ==========================================
    # Build distribution rows for export
    for table, count in palindrome_table_distribution.most_common():
        palindrome_tables_rows.append({"transl_table": table, "palindrome_count": count})

    std_pairs_csv = f"{out_dir}/standard_reversed_pairs.csv"
    palindromes_csv = f"{out_dir}/palindromic_peptides_summary.csv"
    pal_tables_csv = f"{out_dir}/palindrome_table_distribution.csv"
    pal_same15_csv = f"{out_dir}/palindromic_15mers.csv"
    pal_diff15_csv = f"{out_dir}/palindrome_reversed_15mer_pairs.csv"

    pd.DataFrame(std_pairs_rows).to_csv(std_pairs_csv, index=False)
    pd.DataFrame(palindrome_rows).to_csv(palindromes_csv, index=False)
    pd.DataFrame(palindrome_tables_rows).to_csv(pal_tables_csv, index=False)
    pd.DataFrame(pal_same15_rows).to_csv(pal_same15_csv, index=False)
    pd.DataFrame(pal_diff15_rows).to_csv(pal_diff15_csv, index=False)

    print("\nSaved CSV outputs:", flush=True)
    print(f"  - {std_pairs_csv}")
    print(f"  - {palindromes_csv}")
    print(f"  - {pal_tables_csv}")
    print(f"  - {pal_same15_csv}")
    print(f"  - {pal_diff15_csv}")


# --- Execution (local) ---
if __name__ == "__main__":
    # Update this path to point to your local input CSV file
    file_name = "pep_frames_tranls.csv"

    # Update this to the folder where you want output CSVs saved
    output_directory = "pentapeptide_analysis"

    analyze_complete_peptides(file_name, out_dir=output_directory)


In [ ]:
# ============================================================
#  PENTAPEPTIDE ↔ REVERSE PENTAPEPTIDE ANALYSIS
# ============================================================

!pip install pyahocorasick biopython -q

import os
import re
import gc
import hashlib
import pandas as pd
import numpy as np

from collections import defaultdict, Counter
from itertools import product
from tqdm import tqdm

import ahocorasick
from Bio.Data import CodonTable


# ============================================================
# PATHS
# ============================================================

FRAMES_CSV = "/content/final_pep_frames_tranls.csv"
CDS_FASTA = "/content/uniprot_final_CDS.fasta"

OUT_DIR = "/content/pentapeptide_reverse_pair_analysis_mapped_universe"
os.makedirs(OUT_DIR, exist_ok=True)

DEFAULT_TABLE_FOR_INVALID = "1"

CHUNKSIZE = 100_000
WRITE_CHUNK = 20_000


# ============================================================
# OUTPUT FILES
# ============================================================

MASTER_OUT = f"{OUT_DIR}/MASTER_reverse_pentapeptide_CDS_hits_full_details.csv"

ORIGINAL_LEAN_OUT = f"{OUT_DIR}/original_missing_pentapeptide_hits_for_selected_pairs_LEAN.csv"
REVERSE_LEAN_OUT = f"{OUT_DIR}/reverse_pentapeptide_CDS_hits_LEAN.csv"

SUMMARY_OUT = f"{OUT_DIR}/pentapeptide_reverse_pair_cumulative_summary.csv"
REVERSE_FRAME_OUT = f"{OUT_DIR}/reverse_pentapeptide_frame_counts.csv"
REVERSE_TABLE_OUT = f"{OUT_DIR}/reverse_pentapeptide_translation_table_counts.csv"
ORIGINAL_FRAME_OUT = f"{OUT_DIR}/original_missing_pentapeptide_frame_counts.csv"
ORIGINAL_TABLE_OUT = f"{OUT_DIR}/original_missing_pentapeptide_translation_table_counts.csv"

COOCCUR_ALL_OUT = f"{OUT_DIR}/pentapeptide_pairs_cooccurring_same_CDS_all_hits_HASH_ONLY.csv"
COOCCUR_REVERSE_F0_OUT = f"{OUT_DIR}/pentapeptide_pairs_cooccurring_same_CDS_reverse_frame0_HASH_ONLY.csv"
COOCCUR_BOTH_F0_OUT = f"{OUT_DIR}/pentapeptide_pairs_cooccurring_same_CDS_both_frame0_HASH_ONLY.csv"


# ============================================================
# CLEANING FUNCTIONS
# ============================================================

def clean_peptide(x):
    if pd.isna(x):
        return ""
    return str(x).upper().strip()


def clean_dna(x):
    if pd.isna(x):
        return ""

    return (
        str(x)
        .upper()
        .replace("U", "T")
        .replace(" ", "")
        .replace("\n", "")
        .replace("\r", "")
    )


def clean_table(x):
    if pd.isna(x):
        return DEFAULT_TABLE_FOR_INVALID

    x = str(x).strip()

    if x.endswith(".0"):
        x = x.replace(".0", "")

    try:
        int(x)
        return x
    except Exception:
        return DEFAULT_TABLE_FOR_INVALID


def is_valid_table(x):
    try:
        int(str(x))
        return True
    except Exception:
        return False


def sort_tables(tables):
    return sorted([str(t) for t in tables if is_valid_table(t)], key=lambda x: int(x))


def parse_position_first(x):
    if pd.isna(x):
        return np.nan

    nums = re.findall(r"-?\d+\.?\d*", str(x))

    if not nums:
        return np.nan

    return int(float(nums[0]))


def frame_from_start(pos):
    if pd.isna(pos):
        return "Unknown"

    r = int(pos) % 3

    if r == 0:
        return "Frame 0"
    elif r == 1:
        return "Frame +1"
    elif r == 2:
        return "Frame +2"

    return "Unknown"


def hash_text(x):
    return hashlib.blake2b(str(x).encode(), digest_size=8).hexdigest()


def pair_hash(source, rev, cds_hash, table):
    return hashlib.blake2b(
        f"{source}|{rev}|{cds_hash}|{table}".encode(),
        digest_size=8
    ).hexdigest()


def cds_table_hash(cds_hash, table):
    return hashlib.blake2b(
        f"{cds_hash}|{table}".encode(),
        digest_size=8
    ).hexdigest()


# ============================================================
# FASTA STREAMING FUNCTIONS
# ============================================================

def parse_cds_header(header):
    header_clean = header.strip().replace(">", "")
    parts = header_clean.split("|")

    uniprot_id = parts[0].strip() if len(parts) > 0 else ""
    nucleotide_id = parts[1].strip() if len(parts) > 1 else ""
    protein_id = parts[2].strip() if len(parts) > 2 else ""

    molecule_type = parts[3].strip() if len(parts) > 3 else ""
    status = parts[4].strip() if len(parts) > 4 else ""
    uniprot_length = parts[5].strip() if len(parts) > 5 else ""
    ncbi_length = parts[6].strip() if len(parts) > 6 else ""

    transl_table_original = "missing"
    codon_start = "1"

    for part in parts:
        part = part.strip()

        if part.startswith("transl_table="):
            transl_table_original = part.split("=", 1)[1].strip()

        if part.startswith("codon_start="):
            codon_start = part.split("=", 1)[1].strip()

    transl_table_clean = clean_table(transl_table_original)
    was_table_assigned_standard = not is_valid_table(transl_table_original)

    return {
        "uniprot_id": uniprot_id,
        "nucleotide_id": nucleotide_id,
        "protein_id": protein_id,
        "molecule_type": molecule_type,
        "status": status,
        "uniprot_length": uniprot_length,
        "ncbi_length": ncbi_length,
        "transl_table_original": transl_table_original,
        "cds_transl_table": transl_table_clean,
        "codon_start": codon_start,
        "was_transl_table_assigned_standard": was_table_assigned_standard,
        "cds_header": header_clean
    }


def stream_cds_fasta(fasta_path):
    header = None
    seq_parts = []

    with open(fasta_path, "r") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            if line.startswith(">"):
                if header is not None:
                    meta = parse_cds_header(header)
                    seq = clean_dna("".join(seq_parts))

                    if seq:
                        yield meta, seq

                header = line
                seq_parts = []

            else:
                seq_parts.append(line)

        if header is not None:
            meta = parse_cds_header(header)
            seq = clean_dna("".join(seq_parts))

            if seq:
                yield meta, seq


# ============================================================
# CODON FUNCTIONS
# ============================================================

AA_CODON_CACHE = {}


def get_codons_for_aa(aa, transl_table):
    key = (aa, transl_table)

    if key in AA_CODON_CACHE:
        return AA_CODON_CACHE[key]

    table = CodonTable.unambiguous_dna_by_id[int(transl_table)]

    aa_to_codons = defaultdict(list)

    for codon, residue in table.forward_table.items():
        aa_to_codons[residue].append(codon)

    codons = sorted(aa_to_codons.get(aa, []))
    AA_CODON_CACHE[key] = codons

    return codons


def peptide_to_mers_generator(peptide, transl_table):
    codon_lists = []

    for aa in peptide:
        codons = get_codons_for_aa(aa, transl_table)

        if not codons:
            return

        codon_lists.append(codons)

    for combo in product(*codon_lists):
        yield "".join(combo)


# ============================================================
# PASS 1: READ MAPPED PEPTIDE UNIVERSE
# ============================================================

print("\nPASS 1: Reading mapped peptide universe...")

mapped_peptides = set()

usecols_pass1 = ["peptide", "matched_15mer"]

for chunk in tqdm(
    pd.read_csv(FRAMES_CSV, usecols=usecols_pass1, chunksize=CHUNKSIZE, low_memory=False)
):
    chunk["peptide"] = chunk["peptide"].apply(clean_peptide)
    chunk["matched_15mer"] = chunk["matched_15mer"].apply(clean_dna)

    chunk = chunk[
        (chunk["peptide"].str.len() == 5) &
        (chunk["matched_15mer"].str.len() == 15)
    ]

    mapped_peptides.update(chunk["peptide"].unique())

print(f"Unique mapped pentapeptides: {len(mapped_peptides):,}")


# ============================================================
# REVERSE-PAIR UNIVERSE
# ============================================================

mapped_reverse_present = {p for p in mapped_peptides if p[::-1] in mapped_peptides}
mapped_reverse_absent = {p for p in mapped_peptides if p[::-1] not in mapped_peptides}

standard_pairs = set()
palindromes = set()

for p in mapped_peptides:
    r = p[::-1]

    if r in mapped_peptides:
        if p == r:
            palindromes.add(p)
        else:
            standard_pairs.add(tuple(sorted([p, r])))

source_to_reverse = {p: p[::-1] for p in mapped_reverse_absent}
reverse_to_source = {v: k for k, v in source_to_reverse.items()}
reverse_peptides = sorted(reverse_to_source.keys())

print(f"Mapped peptides whose reverse is also mapped: {len(mapped_reverse_present):,}")
print(f"Mapped peptides whose reverse is NOT mapped: {len(mapped_reverse_absent):,}")
print(f"Standard reverse pairs: {len(standard_pairs):,}")
print(f"Palindromes: {len(palindromes):,}")
print(f"Reverse peptides to scan: {len(reverse_peptides):,}")

pd.DataFrame(sorted(standard_pairs), columns=["peptide_A", "peptide_B"]).to_csv(
    f"{OUT_DIR}/mapped_universe_standard_reverse_pairs.csv",
    index=False
)

pd.DataFrame({"palindromic_peptide": sorted(palindromes)}).to_csv(
    f"{OUT_DIR}/mapped_universe_palindromic_peptides.csv",
    index=False
)

pd.DataFrame(
    [
        {
            "source_missing_peptide": k,
            "reverse_peptide_to_scan": v,
            "reverse_is_also_mapped": False
        }
        for k, v in source_to_reverse.items()
    ]
).to_csv(
    f"{OUT_DIR}/target_mapped_missing_peptides_without_mapped_reverse_partner.csv",
    index=False
)


# ============================================================
# PASS 2: ORIGINAL SELECTED MAPPINGS
# ============================================================

print("\nPASS 2: Processing original selected peptide mappings...")

original_pair_hashes = set()
original_frame0_pair_hashes = set()
original_cds_table_hashes = set()

original_frame_counter = Counter()
original_table_counter = Counter()

if os.path.exists(ORIGINAL_LEAN_OUT):
    os.remove(ORIGINAL_LEAN_OUT)

original_header_written = False
original_total_rows = 0
buffer = []

usecols_pass2 = [
    "uniprot_id",
    "nucleotide_id",
    "protein_id",
    "molecule_type",
    "status",
    "uniprot_length",
    "ncbi_length",
    "peptide",
    "matched_15mer",
    "cds_sequence",
    "transl_table",
    "codon_start",
    "num_hits",
    "positions",
    "codon_offset",
    "frame_status",
    "shift"
]

for chunk in tqdm(
    pd.read_csv(FRAMES_CSV, usecols=usecols_pass2, chunksize=CHUNKSIZE, low_memory=False)
):
    chunk["peptide"] = chunk["peptide"].apply(clean_peptide)
    chunk["matched_15mer"] = chunk["matched_15mer"].apply(clean_dna)
    chunk["cds_sequence"] = chunk["cds_sequence"].apply(clean_dna)
    chunk["transl_table"] = chunk["transl_table"].apply(clean_table)

    chunk = chunk[
        (chunk["peptide"].isin(source_to_reverse)) &
        (chunk["peptide"].str.len() == 5) &
        (chunk["matched_15mer"].str.len() == 15) &
        (chunk["cds_sequence"].str.len() > 0)
    ].copy()

    if chunk.empty:
        continue

    chunk["reverse_peptide"] = chunk["peptide"].map(source_to_reverse)
    chunk["start_0based"] = chunk["positions"].apply(parse_position_first)
    chunk["frame"] = chunk["start_0based"].apply(frame_from_start)
    chunk["cds_hash"] = chunk["cds_sequence"].apply(hash_text)

    for row in chunk.itertuples(index=False):
        ph = pair_hash(row.peptide, row.reverse_peptide, row.cds_hash, row.transl_table)
        cth = cds_table_hash(row.cds_hash, row.transl_table)

        original_pair_hashes.add(ph)
        original_cds_table_hashes.add(cth)

        if row.frame == "Frame 0":
            original_frame0_pair_hashes.add(ph)

        original_frame_counter[row.frame] += 1
        original_table_counter[row.transl_table] += 1
        original_total_rows += 1

        buffer.append({
            "source_missing_peptide": row.peptide,
            "reverse_peptide": row.reverse_peptide,
            "source_matched_15mer": row.matched_15mer,
            "source_transl_table": row.transl_table,
            "source_position_raw": row.positions,
            "source_start_0based": row.start_0based,
            "source_frame": row.frame,
            "source_frame_status": row.frame_status,
            "source_shift": row.shift,
            "source_codon_offset": row.codon_offset,
            "source_num_hits": row.num_hits,
            "source_uniprot_id": row.uniprot_id,
            "source_nucleotide_id": row.nucleotide_id,
            "source_protein_id": row.protein_id,
            "source_molecule_type": row.molecule_type,
            "source_status": row.status,
            "source_uniprot_length": row.uniprot_length,
            "source_ncbi_length": row.ncbi_length,
            "source_codon_start": row.codon_start,
            "cds_hash": row.cds_hash,
            "pair_hash": ph
        })

        if len(buffer) >= WRITE_CHUNK:
            pd.DataFrame(buffer).to_csv(
                ORIGINAL_LEAN_OUT,
                mode="a",
                index=False,
                header=not original_header_written
            )
            original_header_written = True
            buffer = []

if buffer:
    pd.DataFrame(buffer).to_csv(
        ORIGINAL_LEAN_OUT,
        mode="a",
        index=False,
        header=not original_header_written
    )

print(f"Original selected mapping rows: {original_total_rows:,}")
print(f"Original unique pair-CDS-table combinations: {len(original_pair_hashes):,}")


# ============================================================
# FASTA TABLE DISCOVERY
# ============================================================

print("\nScanning CDS FASTA once to identify translation tables...")

cds_tables = set()
total_cds_records = 0

for meta, seq in tqdm(stream_cds_fasta(CDS_FASTA)):
    cds_tables.add(meta["cds_transl_table"])
    total_cds_records += 1

unique_tables = sort_tables(cds_tables)

print(f"Total CDS records: {total_cds_records:,}")
print("Translation tables:", unique_tables)


# ============================================================
# REVERSE SCANNING TABLE-BY-TABLE
# ============================================================

reverse_total_hits = 0
reverse_frame_counter = Counter()
reverse_table_counter = Counter()
reverse_peptide_counter = Counter()
source_peptide_counter = Counter()

reverse_unique_cds_table_hashes = set()
reverse_unique_pair_hashes = set()
reverse_frame0_pair_hashes = set()

cooccur_all = set()
cooccur_reverse_frame0 = set()
cooccur_both_frame0 = set()

if os.path.exists(REVERSE_LEAN_OUT):
    os.remove(REVERSE_LEAN_OUT)

if os.path.exists(MASTER_OUT):
    os.remove(MASTER_OUT)

reverse_header_written = False
master_header_written = False

generated_counts = []

for tt in unique_tables:
    print(f"\nProcessing translation table {tt}...")

    A = ahocorasick.Automaton()
    unique_15mer_count = 0
    total_links = 0

    for rev_pep in tqdm(reverse_peptides, desc=f"Building automaton table {tt}"):
        source_pep = reverse_to_source[rev_pep]

        for mer in peptide_to_mers_generator(rev_pep, tt):
            if mer not in A:
                unique_15mer_count += 1

            A.add_word(mer, (source_pep, rev_pep, mer))
            total_links += 1

    A.make_automaton()

    generated_counts.append({
        "transl_table": tt,
        "unique_15mers": unique_15mer_count,
        "total_15mer_pair_links": total_links
    })

    print(f"Table {tt}: {unique_15mer_count:,} unique 15mers")

    reverse_buffer = []
    master_buffer = []

    for meta, cds_seq in tqdm(stream_cds_fasta(CDS_FASTA), desc=f"Scanning CDS table {tt}"):
        cds_tt = meta["cds_transl_table"]

        if cds_tt != tt:
            continue

        cds_hash = hash_text(cds_seq)
        cth = cds_table_hash(cds_hash, tt)
        cds_len = len(cds_seq)

        for end_idx, payload in A.iter(cds_seq):
            source_pep, rev_pep, matched_reverse_15mer = payload

            start_idx = end_idx - 14
            frame = frame_from_start(start_idx)
            is_frame0 = frame == "Frame 0"
            norm_pos = start_idx / cds_len if cds_len else np.nan

            ph = pair_hash(source_pep, rev_pep, cds_hash, tt)

            is_cooccurring_all = ph in original_pair_hashes
            is_cooccurring_reverse_frame0 = is_cooccurring_all and is_frame0
            is_cooccurring_both_frame0 = is_cooccurring_reverse_frame0 and ph in original_frame0_pair_hashes

            reverse_total_hits += 1
            reverse_frame_counter[frame] += 1
            reverse_table_counter[tt] += 1
            reverse_peptide_counter[rev_pep] += 1
            source_peptide_counter[source_pep] += 1

            reverse_unique_cds_table_hashes.add(cth)
            reverse_unique_pair_hashes.add(ph)

            if is_frame0:
                reverse_frame0_pair_hashes.add(ph)

            if is_cooccurring_all:
                cooccur_all.add(ph)

            if is_cooccurring_reverse_frame0:
                cooccur_reverse_frame0.add(ph)

            if is_cooccurring_both_frame0:
                cooccur_both_frame0.add(ph)

            reverse_buffer.append({
                "source_missing_peptide": source_pep,
                "reverse_peptide": rev_pep,
                "matched_reverse_15mer": matched_reverse_15mer,
                "uniprot_id": meta["uniprot_id"],
                "nucleotide_id": meta["nucleotide_id"],
                "protein_id": meta["protein_id"],
                "cds_transl_table": tt,
                "start_0based": start_idx,
                "end_0based": end_idx,
                "start_1based": start_idx + 1,
                "end_1based": end_idx + 1,
                "frame": frame,
                "is_frame0_from_CDS_start": is_frame0,
                "cds_length": cds_len,
                "normalized_position": norm_pos,
                "cds_hash": cds_hash,
                "pair_hash": ph
            })

            master_buffer.append({
                "source_missing_peptide": source_pep,
                "reverse_peptide": rev_pep,
                "matched_reverse_15mer": matched_reverse_15mer,
                "reverse_15mer_length": len(matched_reverse_15mer),
                "reverse_start_0based": start_idx,
                "reverse_end_0based": end_idx,
                "reverse_start_1based": start_idx + 1,
                "reverse_end_1based": end_idx + 1,
                "reverse_frame": frame,
                "reverse_is_frame0_from_CDS_start": is_frame0,
                "reverse_normalized_position": norm_pos,
                "reverse_cds_transl_table": tt,
                "reverse_transl_table_original": meta["transl_table_original"],
                "was_transl_table_assigned_standard": meta["was_transl_table_assigned_standard"],
                "codon_start_from_fasta_header": meta["codon_start"],
                "uniprot_id": meta["uniprot_id"],
                "nucleotide_id": meta["nucleotide_id"],
                "protein_id": meta["protein_id"],
                "molecule_type": meta["molecule_type"],
                "status": meta["status"],
                "uniprot_length": meta["uniprot_length"],
                "ncbi_length": meta["ncbi_length"],
                "cds_length_nt": cds_len,
                "cds_sequence": cds_seq,
                "cds_header": meta["cds_header"],
                "cds_hash": cds_hash,
                "pair_hash": ph,
                "is_cooccurring_with_source_missing_peptide_same_CDS_table": is_cooccurring_all,
                "is_cooccurring_with_source_missing_peptide_and_reverse_frame0": is_cooccurring_reverse_frame0,
                "is_cooccurring_with_source_missing_peptide_both_frame0": is_cooccurring_both_frame0
            })

            if len(reverse_buffer) >= WRITE_CHUNK:
                pd.DataFrame(reverse_buffer).to_csv(
                    REVERSE_LEAN_OUT,
                    mode="a",
                    index=False,
                    header=not reverse_header_written
                )
                reverse_header_written = True
                reverse_buffer = []

            if len(master_buffer) >= WRITE_CHUNK:
                pd.DataFrame(master_buffer).to_csv(
                    MASTER_OUT,
                    mode="a",
                    index=False,
                    header=not master_header_written
                )
                master_header_written = True
                master_buffer = []

    if reverse_buffer:
        pd.DataFrame(reverse_buffer).to_csv(
            REVERSE_LEAN_OUT,
            mode="a",
            index=False,
            header=not reverse_header_written
        )
        reverse_header_written = True

    if master_buffer:
        pd.DataFrame(master_buffer).to_csv(
            MASTER_OUT,
            mode="a",
            index=False,
            header=not master_header_written
        )
        master_header_written = True

    del A
    gc.collect()


# ============================================================
# SAVE SUMMARY OUTPUTS
# ============================================================

pd.DataFrame(generated_counts).to_csv(
    f"{OUT_DIR}/reverse_pentapeptide_generated_15mer_counts_by_table.csv",
    index=False
)

summary_df = pd.DataFrame([
    {
        "mapped_peptide_universe_size": len(mapped_peptides),
        "mapped_reverse_present_members": len(mapped_reverse_present),
        "mapped_reverse_absent_members": len(mapped_reverse_absent),
        "standard_reverse_pairs_in_mapped_universe": len(standard_pairs),
        "palindromic_peptides_in_mapped_universe": len(palindromes),
        "selected_source_missing_peptides_without_mapped_reverse": len(source_to_reverse),
        "unique_reverse_peptides_scanned": len(reverse_peptides),
        "original_mapping_rows_for_selected_sources": original_total_rows,
        "original_unique_pair_CDS_table_combinations": len(original_pair_hashes),
        "original_frame0_pair_CDS_table_combinations": len(original_frame0_pair_hashes),
        "reverse_total_CDS_hits": reverse_total_hits,
        "reverse_unique_reverse_peptides_with_hits": len(reverse_peptide_counter),
        "reverse_unique_source_peptides_with_reverse_hits": len(source_peptide_counter),
        "reverse_unique_CDS_table_pairs": len(reverse_unique_cds_table_hashes),
        "reverse_unique_pair_CDS_table_combinations": len(reverse_unique_pair_hashes),
        "reverse_frame0_pair_CDS_table_combinations": len(reverse_frame0_pair_hashes),
        "cooccurring_pair_CDS_table_all_hits": len(cooccur_all),
        "cooccurring_pair_CDS_table_reverse_frame0": len(cooccur_reverse_frame0),
        "cooccurring_pair_CDS_table_both_frame0": len(cooccur_both_frame0),
        "master_output_file": MASTER_OUT
    }
])

summary_df.to_csv(SUMMARY_OUT, index=False)

pd.DataFrame(
    [{"frame": k, "reverse_hit_count": v} for k, v in reverse_frame_counter.items()]
).to_csv(REVERSE_FRAME_OUT, index=False)

pd.DataFrame(
    [{"transl_table": k, "reverse_hit_count": v} for k, v in reverse_table_counter.items()]
).to_csv(REVERSE_TABLE_OUT, index=False)

pd.DataFrame(
    [{"frame": k, "original_missing_hit_count": v} for k, v in original_frame_counter.items()]
).to_csv(ORIGINAL_FRAME_OUT, index=False)

pd.DataFrame(
    [{"transl_table": k, "original_missing_hit_count": v} for k, v in original_table_counter.items()]
).to_csv(ORIGINAL_TABLE_OUT, index=False)

pd.DataFrame(
    [{"pair_hash": h} for h in cooccur_all]
).to_csv(COOCCUR_ALL_OUT, index=False)

pd.DataFrame(
    [{"pair_hash": h} for h in cooccur_reverse_frame0]
).to_csv(COOCCUR_REVERSE_F0_OUT, index=False)

pd.DataFrame(
    [{"pair_hash": h} for h in cooccur_both_frame0]
).to_csv(COOCCUR_BOTH_F0_OUT, index=False)


# ============================================================
# FINAL PRINT
# ============================================================

print("\n" + "=" * 90)
print("LOW-RAM FULL-MASTER PENTAPEPTIDE REVERSE-PAIR ANALYSIS COMPLETE")
print("=" * 90)

print("\nMapped peptide universe size:")
print(f"{len(mapped_peptides):,}")

print("\nMapped peptides whose reverse is also mapped:")
print(f"{len(mapped_reverse_present):,}")

print("\nMapped peptides whose reverse is NOT mapped:")
print(f"{len(mapped_reverse_absent):,}")

print("\nReverse total CDS hits:")
print(f"{reverse_total_hits:,}")

print("\nCo-occurring source/reverse pairs, all hits:")
print(f"{len(cooccur_all):,}")

print("\nCo-occurring source/reverse pairs, reverse frame 0:")
print(f"{len(cooccur_reverse_frame0):,}")

print("\nCo-occurring source/reverse pairs, both frame 0:")
print(f"{len(cooccur_both_frame0):,}")

print("\nFull master output file:")
print(MASTER_OUT)

print("\nAll outputs saved in:")
print(OUT_DIR)

In [ ]:
# ============================================================
# FINAL PLOTTING CODE WITH CLEAN PAIR-SPECIFIC VENN DIAGRAM
# AND IMPROVED TRANSLATION-TABLE OVERLAP NETWORK
# ============================================================

!pip install matplotlib-venn networkx scipy -q

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib_venn import venn2
from google.colab import drive


# ============================================================
# INPUT / OUTPUT PATHS
# ============================================================

INPUT_DIR = "/content/pentapeptide_reverse_pair_analysis_mapped_universe"

OUTPUT_DIR = "/content/pentapeptide_reverse_pair_analysis_Plots"
PLOT_DIR = f"{OUTPUT_DIR}/final_selected_plots_png_3x4_pair_specific_venn"
os.makedirs(PLOT_DIR, exist_ok=True)


SUMMARY_FILE = f"{INPUT_DIR}/pentapeptide_reverse_pair_cumulative_summary.csv"

REVERSE_FRAME_FILE = f"{INPUT_DIR}/reverse_pentapeptide_frame_counts.csv"
REVERSE_TABLE_FILE = f"{INPUT_DIR}/reverse_pentapeptide_translation_table_counts.csv"

ORIGINAL_FRAME_FILE = f"{INPUT_DIR}/original_missing_pentapeptide_frame_counts.csv"
ORIGINAL_TABLE_FILE = f"{INPUT_DIR}/original_missing_pentapeptide_translation_table_counts.csv"

ORIGINAL_LEAN_FILE = f"{INPUT_DIR}/original_missing_pentapeptide_hits_for_selected_pairs_LEAN.csv"
REVERSE_LEAN_FILE = f"{INPUT_DIR}/reverse_pentapeptide_CDS_hits_LEAN.csv"

MASTER_FILE = f"{INPUT_DIR}/MASTER_reverse_pentapeptide_CDS_hits_full_details.csv"

COOCCUR_ALL_FILE = f"{INPUT_DIR}/pentapeptide_pairs_cooccurring_same_CDS_all_hits_HASH_ONLY.csv"
COOCCUR_REVERSE_F0_FILE = f"{INPUT_DIR}/pentapeptide_pairs_cooccurring_same_CDS_reverse_frame0_HASH_ONLY.csv"
COOCCUR_BOTH_F0_FILE = f"{INPUT_DIR}/pentapeptide_pairs_cooccurring_same_CDS_both_frame0_HASH_ONLY.csv"


# ============================================================
# CHECK FILES
# ============================================================

required_files = [
    SUMMARY_FILE,
    REVERSE_FRAME_FILE,
    REVERSE_TABLE_FILE,
    ORIGINAL_FRAME_FILE,
    ORIGINAL_TABLE_FILE,
    ORIGINAL_LEAN_FILE,
    REVERSE_LEAN_FILE,
    MASTER_FILE,
    COOCCUR_ALL_FILE,
    COOCCUR_REVERSE_F0_FILE,
    COOCCUR_BOTH_F0_FILE
]

missing_files = [f for f in required_files if not os.path.exists(f)]

if missing_files:
    print("Missing files:")
    for f in missing_files:
        print(f)
    raise FileNotFoundError("Some required input files are missing.")

print("Input directory:")
print(INPUT_DIR)

print("\nPlots will be saved in:")
print(PLOT_DIR)


# ============================================================
# STYLE SETTINGS
# ============================================================

DPI = 150
FIGSIZE_3_4 = (9, 12)

plt.rcParams.update({
    "figure.dpi": DPI,
    "savefig.dpi": DPI,
    "font.size": 20,
    "axes.titlesize": 27,
    "axes.labelsize": 24,
    "xtick.labelsize": 19,
    "ytick.labelsize": 19,
    "legend.fontsize": 18,
    "axes.linewidth": 2.0,
    "xtick.major.width": 2.0,
    "ytick.major.width": 2.0,
    "xtick.major.size": 7,
    "ytick.major.size": 7
})

COLORS = {
    "missing": "#5B4E77",
    "reverse": "#2E5E5E",
    "shared": "#4F4F4F",
    "frame0": "#3B556D",
    "frame1": "#7A4F3D",
    "frame2": "#5A6B3F",
    "accent": "#8A6F3D",
    "light_gray": "#B8B3AA",
    "grid": "#D8D3C7",
    "text": "#2B2B2B",
    "missing_light": "#A99EC2",
    "reverse_light": "#8CB5B2",
}

FRAME_ORDER = ["Frame 0", "Frame +1", "Frame +2"]

FRAME_COLORS = {
    "Frame 0": COLORS["frame0"],
    "Frame +1": COLORS["frame1"],
    "Frame +2": COLORS["frame2"]
}


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def style_ax(ax, grid=True):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(2)
    ax.spines["bottom"].set_linewidth(2)

    if grid:
        ax.grid(axis="y", color=COLORS["grid"], linewidth=1.2, alpha=0.75)
        ax.set_axisbelow(True)


def save_png(fig, name):
    path = f"{PLOT_DIR}/{name}.png"
    fig.tight_layout()
    fig.savefig(path, dpi=DPI, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print("Saved:", path)


def compact_int(x):
    return f"{int(float(x)):,}"


def sort_tables(values):
    vals = [str(v).replace(".0", "") for v in values]
    vals = [v for v in vals if v not in ["nan", "None", ""]]
    return sorted(vals, key=lambda x: int(x))


def read_unique_values_in_chunks(file_path, usecol, chunksize=200000):
    values = set()

    for chunk in pd.read_csv(
        file_path,
        usecols=[usecol],
        chunksize=chunksize,
        low_memory=False
    ):
        values.update(chunk[usecol].dropna().astype(str).unique())

    return values


def read_pairhash_to_cdshash_for_hashes(file_path, target_hashes, chunksize=200000):
    cds_hashes = set()

    for chunk in pd.read_csv(
        file_path,
        usecols=["pair_hash", "cds_hash"],
        chunksize=chunksize,
        low_memory=False
    ):
        chunk["pair_hash"] = chunk["pair_hash"].astype(str)
        sub = chunk[chunk["pair_hash"].isin(target_hashes)]

        if not sub.empty:
            cds_hashes.update(sub["cds_hash"].dropna().astype(str).unique())

    return cds_hashes


# ============================================================
# LOAD SUMMARY TABLES
# ============================================================

print("\nLoading summary files...")

summary = pd.read_csv(SUMMARY_FILE).iloc[0].to_dict()

reverse_frame_df = pd.read_csv(REVERSE_FRAME_FILE)
reverse_table_df = pd.read_csv(REVERSE_TABLE_FILE)

original_frame_df = pd.read_csv(ORIGINAL_FRAME_FILE)
original_table_df = pd.read_csv(ORIGINAL_TABLE_FILE)

reverse_frame_df["frame"] = reverse_frame_df["frame"].astype(str)

reverse_frame_df = (
    reverse_frame_df
    .set_index("frame")
    .reindex(FRAME_ORDER)
    .reset_index()
    .fillna(0)
)

if "frame" not in original_frame_df.columns:
    if "frame_from_position" in original_frame_df.columns:
        original_frame_df = original_frame_df.rename(columns={"frame_from_position": "frame"})

original_frame_df["frame"] = original_frame_df["frame"].astype(str)

original_frame_df = (
    original_frame_df
    .set_index("frame")
    .reindex(FRAME_ORDER)
    .reset_index()
    .fillna(0)
)

reverse_table_df["transl_table"] = (
    reverse_table_df["transl_table"]
    .astype(str)
    .str.replace(".0", "", regex=False)
)

original_table_df["transl_table"] = (
    original_table_df["transl_table"]
    .astype(str)
    .str.replace(".0", "", regex=False)
)

print("Summary files loaded.")


# ============================================================
# FIGURE 1: CLEAN PAIR-SPECIFIC VENN DIAGRAM
# ============================================================

print("\nPreparing PAIR-SPECIFIC Venn diagram using cds_hash...")

missing_cds_set = read_unique_values_in_chunks(
    ORIGINAL_LEAN_FILE,
    "cds_hash"
)

reverse_cds_set = read_unique_values_in_chunks(
    REVERSE_LEAN_FILE,
    "cds_hash"
)

cooccur_pair_hashes = read_unique_values_in_chunks(
    COOCCUR_ALL_FILE,
    "pair_hash"
)

pair_specific_shared_cds_set = read_pairhash_to_cdshash_for_hashes(
    ORIGINAL_LEAN_FILE,
    cooccur_pair_hashes
)

pair_specific_shared_cds_set = (
    pair_specific_shared_cds_set
    .intersection(missing_cds_set)
    .intersection(reverse_cds_set)
)

venn_counts = {
    "missing_only": len(missing_cds_set - pair_specific_shared_cds_set),
    "reverse_only": len(reverse_cds_set - pair_specific_shared_cds_set),
    "shared_pair_specific": len(pair_specific_shared_cds_set),
    "missing_total": len(missing_cds_set),
    "reverse_total": len(reverse_cds_set),
    "cooccurring_pair_hashes": len(cooccur_pair_hashes)
}

pd.DataFrame([venn_counts]).to_csv(
    f"{PLOT_DIR}/pair_specific_venn_cds_hash_counts.csv",
    index=False
)

pd.DataFrame(
    {"pair_specific_shared_cds_hash": sorted(pair_specific_shared_cds_set)}
).to_csv(
    f"{PLOT_DIR}/pair_specific_shared_CDS_hashes.csv",
    index=False
)

print("\nPair-specific Venn counts:")
print(venn_counts)

fig, ax = plt.subplots(figsize=(14, 10))

v = venn2(
    subsets=(
        venn_counts["missing_only"],
        venn_counts["reverse_only"],
        venn_counts["shared_pair_specific"]
    ),
    set_labels=("", ""),
    ax=ax
)

if v.get_patch_by_id("10"):
    v.get_patch_by_id("10").set_color(COLORS["missing"])
    v.get_patch_by_id("10").set_alpha(0.75)
    v.get_patch_by_id("10").set_linewidth(2.0)

if v.get_patch_by_id("01"):
    v.get_patch_by_id("01").set_color(COLORS["reverse"])
    v.get_patch_by_id("01").set_alpha(0.75)
    v.get_patch_by_id("01").set_linewidth(2.0)

if v.get_patch_by_id("11"):
    v.get_patch_by_id("11").set_color(COLORS["shared"])
    v.get_patch_by_id("11").set_alpha(0.75)

for text in v.subset_labels:
    if text is not None:
        text.set_fontsize(24)
        text.set_fontweight("bold")
        text.set_color("black")

ax.text(
    0.22,
    -0.08,
    "CDS sequences containing\nmissing pentapeptides",
    transform=ax.transAxes,
    ha="center",
    va="top",
    fontsize=21,
    fontweight="bold",
    color=COLORS["missing"]
)

ax.text(
    0.78,
    -0.08,
    "CDS sequences containing\nrespective reverse pentapeptides",
    transform=ax.transAxes,
    ha="center",
    va="top",
    fontsize=21,
    fontweight="bold",
    color=COLORS["reverse"]
)

ax.set_title(
    "Pair-specific CDS overlap\n"
    "missing pentapeptide P and reverse(P) in same CDS",
    fontsize=28,
    fontweight="bold",
    pad=35
)

plt.subplots_adjust(
    top=0.82,
    bottom=0.22
)

save_png(fig, "figure_01_pair_specific_venn_cds_overlap")


# ============================================================
# FIGURE 2: REVERSE HIT COUNTS BY FRAME
# ============================================================

fig, ax = plt.subplots(figsize=FIGSIZE_3_4)

bars = ax.bar(
    reverse_frame_df["frame"],
    reverse_frame_df["reverse_hit_count"],
    color=[FRAME_COLORS[f] for f in reverse_frame_df["frame"]],
    edgecolor="none"
)

ax.set_ylabel("Number of reverse hits")
ax.set_xlabel("Reading frame")
ax.set_title("Reverse-pentapeptide hits by reading frame")
style_ax(ax, grid=True)

max_val = reverse_frame_df["reverse_hit_count"].max()

for b in bars:
    h = b.get_height()
    ax.text(
        b.get_x() + b.get_width() / 2,
        h + max_val * 0.025,
        compact_int(h),
        ha="center",
        va="bottom",
        fontsize=18,
        fontweight="bold"
    )

save_png(fig, "figure_02_reverse_hits_by_frame")


# ============================================================
# FIGURE 3: REVERSE HIT COUNTS BY TRANSLATION TABLE
# ============================================================

table_plot_df = reverse_table_df.copy()
table_plot_df = table_plot_df.sort_values("reverse_hit_count", ascending=True)

fig, ax = plt.subplots(figsize=FIGSIZE_3_4)

ax.barh(
    table_plot_df["transl_table"],
    table_plot_df["reverse_hit_count"],
    color=COLORS["reverse"],
    edgecolor="none"
)

ax.set_xscale("log")
ax.set_xlabel("Number of reverse hits, log scale")
ax.set_ylabel("Translation table")
ax.set_title("Reverse-pentapeptide hits by translation table")
style_ax(ax, grid=True)

for y, val in enumerate(table_plot_df["reverse_hit_count"]):
    ax.text(
        max(val * 1.12, 1.1),
        y,
        compact_int(val),
        va="center",
        fontsize=15
    )

save_png(fig, "figure_03_reverse_hits_by_translation_table_log")


# ============================================================
# FIGURE 4: TABLE 1 + 11 VERSUS ALL OTHER TABLES
# ============================================================

table_1_11 = reverse_table_df[
    reverse_table_df["transl_table"].isin(["1", "11"])
]["reverse_hit_count"].sum()

other_tables = reverse_table_df[
    ~reverse_table_df["transl_table"].isin(["1", "11"])
]["reverse_hit_count"].sum()

pie_df = pd.DataFrame({
    "group": ["Table 1 + Table 11", "All other tables"],
    "count": [table_1_11, other_tables]
})

fig, ax = plt.subplots(figsize=FIGSIZE_3_4)

ax.pie(
    pie_df["count"],
    labels=pie_df["group"],
    autopct=lambda p: f"{p:.2f}%",
    startangle=90,
    colors=[COLORS["reverse"], COLORS["accent"]],
    textprops={"fontsize": 18, "fontweight": "bold"},
    wedgeprops={"linewidth": 1.5, "edgecolor": "white"}
)

ax.set_title(
    "Dominance of translation tables 1 and 11",
    fontsize=24,
    fontweight="bold"
)

ax.text(
    0,
    -1.25,
    f"Total reverse hits = {compact_int(pie_df['count'].sum())}",
    ha="center",
    fontsize=18
)

save_png(fig, "figure_04_table1_table11_vs_others_pie")


# ============================================================
# FIGURE 5: IMPROVED TRANSLATION TABLE OVERLAP NETWORK
# ============================================================

print("\nPreparing improved translation-table overlap network...")

source_tables = set(original_table_df["transl_table"].astype(str))
reverse_tables = set(reverse_table_df["transl_table"].astype(str))
all_tables = sort_tables(source_tables.union(reverse_tables))

source_count_dict = (
    original_table_df
    .set_index("transl_table")["original_missing_hit_count"]
    .to_dict()
)

reverse_count_dict = (
    reverse_table_df
    .set_index("transl_table")["reverse_hit_count"]
    .to_dict()
)

combined_table_counts = {
    t: source_count_dict.get(t, 0) + reverse_count_dict.get(t, 0)
    for t in all_tables
}

# Dominant tables are automatically selected as top two by combined count.
# In your data this should usually highlight table 1 and table 11.
dominant_tables = set(
    sorted(
        combined_table_counts,
        key=combined_table_counts.get,
        reverse=True
    )[:2]
)

pd.DataFrame([
    {
        "transl_table": t,
        "missing_hit_count": source_count_dict.get(t, 0),
        "reverse_hit_count": reverse_count_dict.get(t, 0),
        "combined_count": combined_table_counts.get(t, 0),
        "present_in_missing": t in source_tables,
        "present_in_reverse": t in reverse_tables,
        "is_dominant_table": t in dominant_tables
    }
    for t in all_tables
]).to_csv(
    f"{PLOT_DIR}/translation_table_network_node_summary.csv",
    index=False
)

fig, ax = plt.subplots(figsize=(12, 16))
ax.axis("off")

# Better spacing so side circles are not clipped
left_x = 0.16
mid_x = 0.50
right_x = 0.84

missing_node = (left_x, 0.50)
reverse_node = (right_x, 0.50)

table_y = np.linspace(0.88, 0.12, len(all_tables))
table_pos = {t: (mid_x, y) for t, y in zip(all_tables, table_y)}

max_source = max(source_count_dict.values()) if len(source_count_dict) > 0 else 1
max_reverse = max(reverse_count_dict.values()) if len(reverse_count_dict) > 0 else 1

# Edges from missing peptides to tables
for t in all_tables:
    if t in source_tables:
        is_dom = t in dominant_tables
        raw_count = source_count_dict.get(t, 0)

        lw = 2.0 + 7.0 * (raw_count / max_source)
        if is_dom:
            lw = max(lw, 7.0)

        ax.plot(
            [missing_node[0], table_pos[t][0]],
            [missing_node[1], table_pos[t][1]],
            color=COLORS["missing"],
            linewidth=lw,
            alpha=0.85 if is_dom else 0.42,
            zorder=1
        )

# Edges from reverse peptides to tables
for t in all_tables:
    if t in reverse_tables:
        is_dom = t in dominant_tables
        raw_count = reverse_count_dict.get(t, 0)

        lw = 2.0 + 7.0 * (raw_count / max_reverse)
        if is_dom:
            lw = max(lw, 7.0)

        ax.plot(
            [reverse_node[0], table_pos[t][0]],
            [reverse_node[1], table_pos[t][1]],
            color=COLORS["reverse"],
            linewidth=lw,
            alpha=0.85 if is_dom else 0.42,
            zorder=1
        )

# Large side nodes
side_node_size = 11500

ax.scatter(
    missing_node[0],
    missing_node[1],
    s=side_node_size,
    color=COLORS["missing"],
    edgecolor="white",
    linewidth=2.5,
    zorder=5
)

ax.text(
    missing_node[0],
    missing_node[1],
    "Missing\npeptides",
    ha="center",
    va="center",
    fontsize=15,
    fontweight="bold",
    color="white",
    zorder=6
)

ax.scatter(
    reverse_node[0],
    reverse_node[1],
    s=side_node_size,
    color=COLORS["reverse"],
    edgecolor="white",
    linewidth=2.5,
    zorder=5
)

ax.text(
    reverse_node[0],
    reverse_node[1],
    "Reverse\npeptides",
    ha="center",
    va="center",
    fontsize=15,
    fontweight="bold",
    color="white",
    zorder=6
)

# Translation table nodes
for t in all_tables:
    x, y = table_pos[t]

    in_missing = t in source_tables
    in_reverse = t in reverse_tables
    shared = in_missing and in_reverse
    dominant = t in dominant_tables

    if shared:
        facecolor = COLORS["shared"]
        textcolor = "white"
    elif in_missing:
        facecolor = COLORS["missing_light"]
        textcolor = COLORS["text"]
    elif in_reverse:
        facecolor = COLORS["reverse_light"]
        textcolor = COLORS["text"]
    else:
        facecolor = COLORS["light_gray"]
        textcolor = COLORS["text"]

    node_size = 1750 if not dominant else 2350

    ax.scatter(
        x,
        y,
        s=node_size,
        color=facecolor,
        edgecolor="white",
        linewidth=2.0,
        zorder=5
    )

    ax.text(
        x,
        y,
        t,
        ha="center",
        va="center",
        fontsize=18 if not dominant else 20,
        fontweight="bold",
        color=textcolor,
        zorder=6
    )

# Side labels
ax.text(
    missing_node[0],
    0.035,
    "Missing\npentapeptide group",
    ha="center",
    va="center",
    fontsize=15,
    fontweight="bold",
    color=COLORS["missing"]
)

ax.text(
    reverse_node[0],
    0.035,
    "Reverse\npentapeptide group",
    ha="center",
    va="center",
    fontsize=15,
    fontweight="bold",
    color=COLORS["reverse"]
)

# Title and subtitle
ax.text(
    0.5,
    0.975,
    "Translation-table overlap network",
    ha="center",
    va="center",
    fontsize=28,
    fontweight="bold",
    color=COLORS["text"]
)

ax.text(
    0.5,
    0.935,
    "Dominant translation tables are highlighted by thicker edges",
    ha="center",
    va="center",
    fontsize=17,
    color=COLORS["text"]
)

# Legend-style explanation
ax.text(
    0.5,
    0.005,
    "Dark table nodes are shared by both groups; coloured light nodes are group-specific. "
    "Thicker edges indicate dominant translation-table associations.",
    ha="center",
    va="bottom",
    fontsize=13,
    color=COLORS["text"]
)

ax.set_xlim(0.00, 1.00)
ax.set_ylim(0.00, 1.00)

save_png(fig, "figure_05_translation_table_overlap_network_improved")


# ============================================================
# FIGURE 6: CO-OCCURRENCE SUMMARY
# ============================================================

cooccur_summary_df = pd.DataFrame({
    "category": [
        "Missing peptide\nCDS-table pairs",
        "Reverse peptide\nCDS-table pairs",
        "Co-occurring\nall hits",
        "Co-occurring\nreverse frame 0",
        "Co-occurring\nboth frame 0"
    ],
    "count": [
        summary["original_unique_pair_CDS_table_combinations"],
        summary["reverse_unique_pair_CDS_table_combinations"],
        summary["cooccurring_pair_CDS_table_all_hits"],
        summary["cooccurring_pair_CDS_table_reverse_frame0"],
        summary["cooccurring_pair_CDS_table_both_frame0"]
    ]
})

fig, ax = plt.subplots(figsize=FIGSIZE_3_4)

bars = ax.bar(
    cooccur_summary_df["category"],
    cooccur_summary_df["count"],
    color=[
        COLORS["missing"],
        COLORS["reverse"],
        COLORS["shared"],
        COLORS["accent"],
        "#222222"
    ],
    edgecolor="none"
)

ax.set_yscale("log")
ax.set_ylabel("Count, log scale")
ax.set_title("Source/reverse co-occurrence summary")
style_ax(ax, grid=True)

for b in bars:
    h = b.get_height()
    ax.text(
        b.get_x() + b.get_width() / 2,
        h * 1.20 if h > 0 else 1,
        compact_int(h),
        ha="center",
        va="bottom",
        fontsize=15,
        fontweight="bold"
    )

plt.xticks(rotation=35, ha="right")

save_png(fig, "figure_06_cooccurrence_summary_log")


# ============================================================
# FIGURE 7: CO-OCCURRENCE FRAME COMBINATION HEATMAP
# ============================================================

print("\nPreparing co-occurrence frame-combination heatmap using pair_hash...")

cooccur_hash_df = pd.read_csv(COOCCUR_ALL_FILE)

original_frame_hash_df = pd.read_csv(
    ORIGINAL_LEAN_FILE,
    usecols=["pair_hash", "source_frame"]
).drop_duplicates()

reverse_frame_hash_df = pd.read_csv(
    REVERSE_LEAN_FILE,
    usecols=["pair_hash", "frame"]
).rename(columns={"frame": "reverse_frame"}).drop_duplicates()

cooccur_hash_df["pair_hash"] = cooccur_hash_df["pair_hash"].astype(str)
original_frame_hash_df["pair_hash"] = original_frame_hash_df["pair_hash"].astype(str)
reverse_frame_hash_df["pair_hash"] = reverse_frame_hash_df["pair_hash"].astype(str)

cooccur_frames = (
    cooccur_hash_df
    .merge(original_frame_hash_df, on="pair_hash", how="left")
    .merge(reverse_frame_hash_df, on="pair_hash", how="left")
)

frame_matrix = (
    cooccur_frames
    .groupby(["source_frame", "reverse_frame"])
    .size()
    .reset_index(name="count")
)

heat = (
    frame_matrix
    .pivot(index="source_frame", columns="reverse_frame", values="count")
    .reindex(index=FRAME_ORDER, columns=FRAME_ORDER)
    .fillna(0)
)

heat.to_csv(
    f"{PLOT_DIR}/cooccurrence_frame_combination_matrix.csv"
)

fig, ax = plt.subplots(figsize=FIGSIZE_3_4)

vals = heat.values

im = ax.imshow(vals, cmap="Oranges", aspect="auto")

ax.set_xticks(np.arange(len(FRAME_ORDER)))
ax.set_xticklabels(FRAME_ORDER, rotation=25, ha="right")
ax.set_yticks(np.arange(len(FRAME_ORDER)))
ax.set_yticklabels(FRAME_ORDER)

ax.set_xlabel("Reverse frame")
ax.set_ylabel("Missing peptide frame")
ax.set_title(
    "Co-occurring missing/reverse pairs by frame",
    fontsize=23,
    fontweight="bold"
)

for i in range(vals.shape[0]):
    for j in range(vals.shape[1]):
        ax.text(
            j,
            i,
            compact_int(vals[i, j]),
            ha="center",
            va="center",
            fontsize=19,
            fontweight="bold",
            color=COLORS["text"]
        )

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Number of co-occurring pairs", fontsize=17)

save_png(fig, "figure_07_cooccurrence_frame_combination_heatmap")


# ============================================================
# SAVE PLOTTING SUMMARY
# ============================================================

plot_summary = {
    "input_dir": INPUT_DIR,
    "output_dir": OUTPUT_DIR,
    "plot_dir": PLOT_DIR,
    "missing_cds_total": venn_counts["missing_total"],
    "reverse_cds_total": venn_counts["reverse_total"],
    "pair_specific_shared_cds_total": venn_counts["shared_pair_specific"],
    "missing_only_cds_pair_specific_venn": venn_counts["missing_only"],
    "reverse_only_cds_pair_specific_venn": venn_counts["reverse_only"],
    "cooccurring_pair_hashes": venn_counts["cooccurring_pair_hashes"],
    "mapped_peptide_universe_size": summary["mapped_peptide_universe_size"],
    "mapped_reverse_present_members": summary["mapped_reverse_present_members"],
    "mapped_reverse_absent_members": summary["mapped_reverse_absent_members"],
    "reverse_total_CDS_hits": summary["reverse_total_CDS_hits"],
    "cooccurring_pair_CDS_table_all_hits": summary["cooccurring_pair_CDS_table_all_hits"],
    "cooccurring_pair_CDS_table_reverse_frame0": summary["cooccurring_pair_CDS_table_reverse_frame0"],
    "cooccurring_pair_CDS_table_both_frame0": summary["cooccurring_pair_CDS_table_both_frame0"],
    "table_1_11_reverse_hits": table_1_11,
    "other_table_reverse_hits": other_tables,
    "table_1_11_percent": 100 * table_1_11 / (table_1_11 + other_tables)
}

pd.DataFrame([plot_summary]).to_csv(
    f"{PLOT_DIR}/plotting_master_summary.csv",
    index=False
)


# ============================================================
# FINAL MESSAGE
# ============================================================

print("\n" + "=" * 90)
print("PAIR-SPECIFIC VENN PENTAPEPTIDE REVERSE-PAIR PLOTTING COMPLETE")
print("=" * 90)

print("\nInput CSV directory:")
print(INPUT_DIR)

print("\nPlots saved in:")
print(PLOT_DIR)

print("\nGenerated PNG figures:")
for f in sorted(os.listdir(PLOT_DIR)):
    if f.endswith(".png"):
        print(" -", f)